In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/home/studinstru/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Adding W&B API key

In [ ]:
# Secrets: works BOTH on Kaggle and on a local machine.
# Never hardcode a token in this notebook - it is committed to git.
#   Kaggle -> Add-ons > Secrets  (names: WANDB_API_KEY, HF_TOKEN)
#   Local  -> copy .env.example to .env at the repo root and fill it in
import os

try:
    from kaggle_secrets import UserSecretsClient          # only exists on Kaggle
    _s = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = _s.get_secret("WANDB_API_KEY")
    os.environ["HF_TOKEN"] = _s.get_secret("HF_TOKEN")
    print("Secrets loaded from Kaggle Secrets.")
except ImportError:
    from dotenv import load_dotenv, find_dotenv           # pip install python-dotenv
    load_dotenv(find_dotenv(usecwd=True))                 # finds .env at the repo root
    print("Secrets loaded from .env (local run).")

WB_KEY = os.environ.get("WANDB_API_KEY")

_missing = [k for k in ("WANDB_API_KEY", "HF_TOKEN") if not os.environ.get(k)]
if _missing:
    print("WARNING: missing secret(s):", ", ".join(_missing))
    print("  W&B logging and/or the Hugging Face push will be skipped or fail.")


In [ ]:
import wandb

wandb.login(key=WB_KEY)

In [ ]:
!pip install sentence-transformers

In [ ]:
import wandb

print("Initializing wandb")

run = wandb.init(
    entity="ambatwar-iitm",
    project="dlgenai-t226",
    name="mcq-solver-full-pipeline",
    group="full-pipeline-runs",
    job_type="full_pipeline",
    tags=["all_models", "training_curves"],
    config={
        "task": "smart-mcq-solver-challenge",
    },
)

print("W&B run:", run.url)
print("This run tracks step-by-step training curves and artifacts for the whole notebook.")
print("A separate run is logged per model near the end (Step 13) so all models can be")
print("compared side by side in the W&B UI (Runs table / bar chart / parallel coordinates).")


## Step 1 - Setup & environment check

In [ ]:
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

import matplotlib.pyplot as plt

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.system(), platform.release())
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("-" * 45)

# The device we'll use for all tensor computation.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device      :", device)
if device == "cuda":
    print("GPU         :", torch.cuda.get_device_name(0))
    print("VRAM (GB)   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("-" * 45)

# Confirm the dataset files exist (paths are relative to this notebook's folder).
DATA_DIR = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
for fname in ["train.csv", "test.csv", "sample_submission.csv"]:
    fpath = DATA_DIR / fname
    status = "FOUND   " if fpath.exists() else "MISSING!"
    print(f"{fname:22s} {status} -> {fpath}")

#import os
#from kaggle_secrets import UserSecretsClient
#os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## Step 2 - Reading input

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

train.head()

In [ ]:
test.head()

In [ ]:
train.describe()

In [ ]:
# check the shape of train and test
print(train.shape)
print(test.shape)
print(sample_sub.shape)

In [ ]:
print("train columns:", list(train.columns))
print("test columns :", list(test.columns))
print()

print("Missing values per column (train):")
print(train.isna().sum().to_string())
print()

In [ ]:
def show_question(row, show_answer=True):
    """ Preety-print a single question row so it is easy to read. """
    print("PROMPT:")
    print("  " + str(row["prompt"]))
    print()
    for opt in ["A", "B", "C", "D", "E"]:
        print(f"  {opt}) {row[opt]}")
    if show_answer and "answer" in row.index:
        print()
        print(" CORRECT ANSWER:", row["answer"])

print("=" * 80)
print("ONE EXAMPLE FROM THE TRAINING SET")
print("=" * 80)
show_question(train.iloc[0])
print("=" *80)

## Step 3 - Explore the data (EDA)

EDA stands for Exploratory Data Analysis. Before any modeling we ask simple questions about the data, because the answers change every later decision:

1. **How are the correct answers distributed across A, B, C, D, E?**
   - If they are roughly equal, there is no shortcut: a constant guess like "A B C" cannot beat chance.
   - If one letter is much more common, even a simple baseline can exploit that bias.
2. **How long are the prompts and options (in words)?**
   - This sets the maximum sequence length we feed a transformer later, which directly drives GPU memory use.

Run the cell below.

In [ ]:
train["answer"]

In [ ]:
# === Step 3: Explore the data (EDA) ===
import matplotlib.pyplot as plt

option_cols = ["A", "B", "C", "D", "E"]

# --- 1) Distribution of the correct answer label ---
answer_counts = train["answer"].value_counts().reindex(option_cols).fillna(0).astype(int)
answer_percent = (train["answer"].value_counts(normalize=True).reindex(option_cols).fillna(0) * 100).round(1)
print("Correct answer distribution in train (out of 2000):")
for label in option_cols:
    print(f" {label}: {answer_counts[label]:4d} ({answer_percent[label]}%)")
print()

# --- 2) Text length analysis (counted in words) ---
train["prompt_words"] = train["prompt"].str.split().str.len()
train["max_option_words"] = train[option_cols].apply(
    lambda r: max(len(str(x).split()) for x in r), axis=1
)

print("Prompt length in words:")
print(train["prompt_words"].describe().round(1).to_string())
print()
print("Longest option length in words (per question):")
print(train["max_option_words"].describe().round(1).to_string())

# --- 3) Visualize ---
figure, axes = plt.subplots(1, 2, figsize = (11, 4))
axes[0].bar(answer_counts.index, answer_counts.values, color="steelblue")
axes[0].set_xlabel("answer label")
axes[0].set_ylabel("count")

axes[1].hist(train["prompt_words"], bins=40, color="red")
axes[1].set_title("Prompt length (words)")
axes[1].set_xlabel("words per prompt")


It's mildly imbalanced. B and C come up more often; E least. The data isn't perfectly flat, so the smartest constant guess is the three most frequent letters in order: "B C A", not "A B C". That's a small but free edge we'll measure in a moment. The key lesson: a model still has to actually read the question - you can't win on label bias alone here.

Prompt length (right chart): most prompts are 10-30 words (peak ~15), tailing off by ~50. Prompts are short.

## Step 4 - Build a local scoreboard (metric + validation split + baselines)

We cannot see the test answers, so we need our own honest way to measure progress. Three pieces:

1. **The MAP@3 metric, written by hand.** With one correct answer per question it is simply `1 / (rank of the correct label)` if that label lands in our top 3, otherwise 0. We verify it against the exact example from the problem statement.
2. **A leakage-free validation set.** A plain stratified 20% holdout is not safe on this dataset: many questions are near-duplicates of each other, so a random split leaves a copy of most validation questions sitting in train, and the score measures recall rather than reasoning. Instead we embed every prompt with MiniLM, join any pair with cosine similarity >= 0.95 into a cluster, and assign **whole clusters** to one side of the split. The cell measures the difference and plots it: the share of validation questions that still have a near-duplicate in train, for a random split versus this one.
3. **Three baselines.** Two are constant guesses - always "A B C", and always the three most frequent answers - which measure how much of a score comes from the label distribution alone.

   The third is more important and more uncomfortable. This dataset builds every question by taking one text from a pool of correct answers and four from a separate pool of distractors, and **the two pools never overlap**: only ~313 distinct texts ever appear as a correct answer, and not one of them is ever reused as a wrong option. An option can therefore be identified as correct by recognising its text, without reading the question at all. The third baseline does precisely that and nothing more.

   It matters because it changes what "good" means. If a question-blind rule already scores highly, then the gap between it and a trained model is the only part of that model's score attributable to actually reading the question. Reporting a result against the 0.39 constant baseline would credit the model with work the dataset did for it.

Important idea - **data leakage**: we compute the "most frequent" order from the *training split only*, never from validation. Using the validation data to make our predictions would flatter the score dishonestly.

Run the cell below.



In [ ]:
# === Step 4: Local scoreboard - MAP@3 metric, validation split, constant baselines ===
from sklearn.model_selection import train_test_split


# 1) Write the MAP@3 metric ourselves (exactly one correct label per question).
def average_precision_at_3(true_label, predicted_labels):
    """Score one question: 1.0 if correct is 1st, 0.5 if 2nd, 1/3 if 3rd, else 0.0."""
    for rank, label in enumerate(predicted_labels[:3], start=1):
        if label == true_label:
            return 1.0 / rank
    return 0.0


def map_at_3(true_labels, predicted_lists):
    """Mean of the per-question scores over the whole set."""
    scores = [average_precision_at_3(t, p) for t, p in zip(true_labels, predicted_lists)]
    return sum(scores) / len(scores)


# Sanity check against the example in the problem statement (correct answer = A):
assert average_precision_at_3("A", ["A", "B", "C"]) == 1.0
assert average_precision_at_3("A", ["B", "A", "C"]) == 0.5
assert abs(average_precision_at_3("A", ["C", "D", "A"]) - 1 / 3) < 1e-9
assert average_precision_at_3("A", ["C", "D", "E"]) == 0.0
print("MAP@3 metric passes the sanity checks.\n")

# 2) Hold out 20% of train as a validation set - leakage-safe this time.
# A naive random split can place near-duplicate/templated questions on both
# sides of the boundary. Step 11's own diagnostic later found exactly that:
# with a plain random split, 82.8% of valid_split had a near-identical
# neighbor still sitting in train_split, which lets retrieval-based methods
# copy the answer instead of reasoning and inflates their score without
# meaning better generalization. Fix: cluster near-duplicate prompts first,
# then split whole CLUSTERS between train_split/valid_split instead of
# individual rows, so no cluster is ever split across the boundary.
#
# An earlier version of this clustering used TF-IDF (word-overlap) similarity,
# which only caught duplicates that reuse the same wording and left 55.8% of
# valid_split still near-duplicated (per Step 11's check) - because two
# questions can mean almost the same thing while using different words, which
# TF-IDF misses but MiniLM embeddings catch. RAG's retrieval is itself MiniLM
# embeddings, so clustering with the same method Step 11 uses to detect
# leakage closes that gap: verified 0/400 (0.0%) near-duplicates after this.
from sentence_transformers import SentenceTransformer as _DedupSentenceTransformer

DEDUP_SIM_THRESHOLD = 0.95  # keep in sync with Step 11's NEAR_DUP_THRESHOLD

train = train.reset_index(drop=True)
_dedup_model = _DedupSentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
_dedup_embs = _dedup_model.encode(
    train["prompt"].astype(str).tolist(), convert_to_numpy=True,
    normalize_embeddings=True, show_progress_bar=False, batch_size=64,
)
_sim_matrix = _dedup_embs @ _dedup_embs.T  # embeddings are normalized, so dot product = cosine similarity
del _dedup_model
if device == "cuda":
    torch.cuda.empty_cache()

# Union-find: group every question with cosine similarity >= threshold into one cluster.
_parent = list(range(len(train)))

def _find(i):
    while _parent[i] != i:
        _parent[i] = _parent[_parent[i]]
        i = _parent[i]
    return i

def _union(i, j):
    ri, rj = _find(i), _find(j)
    if ri != rj:
        _parent[ri] = rj

for i in range(len(train)):
    row_sims = _sim_matrix[i]
    for j in range(i + 1, len(train)):
        if row_sims[j] >= DEDUP_SIM_THRESHOLD:
            _union(i, j)

train["_cluster"] = [_find(i) for i in range(len(train))]
n_clusters = train["_cluster"].nunique()
print(f"Near-duplicate clustering: {len(train)} questions -> {n_clusters} clusters "
      f"(threshold cosine >= {DEDUP_SIM_THRESHOLD})")

# Assign whole clusters to valid_split (shuffled, fixed seed) until ~20% of rows is reached,
# so every near-duplicate cluster lands entirely on one side of the split.
_cluster_sizes = train.groupby("_cluster").size().sample(frac=1, random_state=42)
_target_valid_rows = int(round(0.2 * len(train)))
_valid_clusters, _running = set(), 0
for _cid, _size in _cluster_sizes.items():
    if _running >= _target_valid_rows:
        break
    _valid_clusters.add(_cid)
    _running += _size

# ---- Measure the leakage this split removes, so the fix is evidence, not a claim ----
# For each validation question, is there a near-duplicate sitting on the training
# side? A plain random split leaves most of them there; the cluster split cannot,
# because whole clusters move together. Step 13 turns this into a bar chart.
def _near_duplicate_rate(valid_idx, train_idx):
    if len(valid_idx) == 0 or len(train_idx) == 0:
        return 0.0
    best_match = _sim_matrix[np.ix_(valid_idx, train_idx)].max(axis=1)
    return float((best_match >= DEDUP_SIM_THRESHOLD).mean())

_cluster_is_valid = train["_cluster"].isin(_valid_clusters).to_numpy()
_rng_leak = np.random.default_rng(42)
_rand_valid_idx = _rng_leak.choice(len(train), size=_target_valid_rows, replace=False)
_rand_is_valid = np.zeros(len(train), dtype=bool)
_rand_is_valid[_rand_valid_idx] = True

LEAKAGE_STATS = {
    "random split (leaky)": _near_duplicate_rate(np.where(_rand_is_valid)[0], np.where(~_rand_is_valid)[0]),
    "cluster split (used)": _near_duplicate_rate(np.where(_cluster_is_valid)[0], np.where(~_cluster_is_valid)[0]),
}
print("Validation questions having a near-duplicate on the training side:")
for _name, _rate in LEAKAGE_STATS.items():
    print(f"  {_name:22s} {_rate:6.1%}")

del _sim_matrix, _dedup_embs   # ~24 MB of pairwise similarities, no longer needed

valid_split = train[train["_cluster"].isin(_valid_clusters)].drop(columns="_cluster").reset_index(drop=True)
train_split = train[~train["_cluster"].isin(_valid_clusters)].drop(columns="_cluster").reset_index(drop=True)
# Keep the cluster label per row before dropping the column, so Step 4.5 can show
# the duplicate structure of the dataset without re-running the embedding step.
CLUSTER_LABELS = train["_cluster"].to_numpy().copy()
train = train.drop(columns="_cluster")

valid_true = valid_split["answer"].tolist()
print(f"train_split: {len(train_split)} rows   valid_split: {len(valid_split)} rows")
# Cluster-based splitting can't guarantee perfect label balance the way stratified
# random splitting does, so print the actual distribution to confirm it isn't skewed.
print("Answer distribution - train_split:", train_split["answer"].value_counts().to_dict())
print("Answer distribution - valid_split:", valid_split["answer"].value_counts().to_dict())
print()

# 3) Constant baselines. The 'frequency' order is learned ONLY from train_split (no leakage).
freq_order = train_split["answer"].value_counts().index.tolist()[:3]
print("Three most frequent answers in train_split:", freq_order, "\n")

baselines = {
    "constant 'A B C'": ["A", "B", "C"],
    f"constant '{' '.join(freq_order)}' (most frequent)": freq_order,
}

scoreboard = {}
# Per-epoch curves are logged live to W&B, but they are also kept here so Step 13
# can draw every trained model on ONE shared chart instead of four separate ones.
TRAINING_HISTORY = {}

# ---- Figure helpers -------------------------------------------------------
# Each result is plotted at the step that produces it, so the notebook output
# reads as a story rather than a wall of charts at the end. Every figure is also
# written to figures/ as a PNG so the report can embed it directly.
import os

SAVED_FIGURES = []


def save_and_show(fig, filename):
    """Write the figure to figures/ and render it in the cell output."""
    os.makedirs("figures", exist_ok=True)
    path = os.path.join("figures", filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if path not in SAVED_FIGURES:
        SAVED_FIGURES.append(path)
    plt.show()
    print(f"  saved {path}")


def show_training_curve(model_name, filename):
    """Draw one model's per-epoch training loss and validation MAP@3."""
    hist = TRAINING_HISTORY.get(model_name)
    if not hist or not hist["epoch"]:
        print(f"  no recorded history for {model_name}")
        return
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", color="#d1495b")
    axes[0].set_ylabel("training loss")
    axes[0].set_title("Training loss")
    axes[1].plot(hist["epoch"], hist["val_map_at_3"], marker="o", color="#2a9d8f")
    axes[1].set_ylabel("MAP@3")
    axes[1].set_title("Validation MAP@3")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.set_xticks(hist["epoch"])       # epochs are whole numbers
        ax.grid(alpha=0.3)
    fig.suptitle(model_name)
    fig.tight_layout()
    save_and_show(fig, filename)


def show_confusion_matrix(logits, true_letters, letters, title, filename):
    """Confusion matrix over the option letters: does the model favour a position?"""
    true_idx = [letters.index(a) for a in true_letters]
    pred_idx = [int(j) for j in logits.argmax(axis=1)]
    matrix = np.zeros((len(letters), len(letters)), dtype=int)
    for t, p in zip(true_idx, pred_idx):
        matrix[t, p] += 1
    top1 = sum(t == p for t, p in zip(true_idx, pred_idx)) / len(true_idx)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(letters)), letters)
    ax.set_yticks(range(len(letters)), letters)
    ax.set_xlabel("predicted option")
    ax.set_ylabel("true option")
    ax.set_title(f"{title}\ntop-1 accuracy {top1:.4f}")
    for r in range(len(letters)):
        for c in range(len(letters)):
            ax.text(c, r, matrix[r, c], ha="center", va="center", fontsize=9,
                    color="white" if matrix[r, c] > matrix.max() / 2 else "black")
    fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout()
    save_and_show(fig, filename)
    return top1


# ---- The leakage this split removes, plotted where it is measured ----
_fig, _ax = plt.subplots(figsize=(6.4, 3.4))
_bars = _ax.bar(list(LEAKAGE_STATS), list(LEAKAGE_STATS.values()), color=["#d1495b", "#2a9d8f"])
for _bar, _rate in zip(_bars, LEAKAGE_STATS.values()):
    _ax.text(_bar.get_x() + _bar.get_width() / 2, _rate + 0.02, f"{_rate:.1%}",
             ha="center", fontsize=10)
_ax.set_ylabel("share of validation questions")
_ax.set_ylim(0, 1.05)
_ax.set_title("Validation questions with a near-duplicate in train")
_ax.grid(axis="y", alpha=0.3)
_fig.tight_layout()
save_and_show(_fig, "fig_leakage_before_after.png")
for name, guess in baselines.items():
    preds = [guess] * len(valid_split)          # same guess for every validation question
    scoreboard[name] = map_at_3(valid_true, preds)

# ---- A third baseline: the answer-text artifact ----
# This dataset draws correct answers and distractors from two pools that never
# overlap. Across all 2,000 questions only ~313 distinct texts ever appear as the
# correct answer, and not one of them is ever reused as a wrong option. So an
# option can be identified as correct purely by recognising its text, with no
# reference to the question being asked.
#
# The model below does exactly that and nothing else: for each option it asks
# "was this exact text a correct answer somewhere in train_split?" and ranks the
# matches first. It never reads the prompt. Whatever it scores is the share of
# every later result that comes from the artifact rather than from comprehension,
# which makes it the honest bar to clear - not the 0.39 constant baseline.
_known_answer_texts = {row[row["answer"]] for _, row in train_split.iterrows()}
_letter_priority = {letter: rank for rank, letter in
                    enumerate(train_split["answer"].value_counts().index.tolist())}


def _artifact_rank(row):
    """Rank options by whether their text was ever a correct answer in train_split.
    Ties (the usual case for the four distractors) fall back to letter frequency."""
    return sorted(option_cols,
                  key=lambda c: (-(row[c] in _known_answer_texts),
                                 _letter_priority.get(c, len(option_cols))))


artifact_preds = [_artifact_rank(row)[:3] for _, row in valid_split.iterrows()]
ARTIFACT_BASELINE = "Answer-text artifact (never reads the question)"
scoreboard[ARTIFACT_BASELINE] = map_at_3(valid_true, artifact_preds)

_single_match = sum(
    1 for _, row in valid_split.iterrows()
    if sum(row[c] in _known_answer_texts for c in option_cols) == 1
)
print(f"Answer-text artifact: {len(_known_answer_texts)} distinct correct-answer texts in train_split;")
print(f"  {_single_match}/{len(valid_split)} validation questions have exactly ONE option carrying one.")
print(f"  Scoring by that signal alone gives MAP@3 {scoreboard[ARTIFACT_BASELINE]:.4f} without reading anything.\n")

print("Local validation MAP@3 (higher is better):")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")

wandb.log(scoreboard)


Conclusion: the two constant baselines score 0.3937 ("A B C") and 0.4088
("B C D", the three most frequent answers in train_split). The gap between them
is the entire value of knowing the label distribution - about 1.5 points. Every
model from here on has to beat 0.4088 to be worth anything, and anything close
to it is not reading the question at all.


## Step 4.5 - Where the near-duplicate numbers come from

Step 4 asserted that 2,000 questions collapse into 613 clusters and built the entire
validation strategy on that. A claim carrying that much weight should be inspectable,
so this step derives it:

1. **The size distribution** of the 613 clusters, which shows how the dataset is
   really composed, grouped into bands from unique to heavily repeated.
2. **Unique against duplicated**, separating questions that have no twin from those
   that do.
3. **Effective dataset size**, which is the practical consequence: 2,000 rows are
   only 613 distinct questions, so the row count overstates the available material by
   more than three times. The same count is reported for each side of the split.
4. **The largest cluster printed in full**, so the grouping can be judged by eye
   rather than trusted.

The fourth is the one to look at. It shows what "near-duplicate" means concretely here,
and why a random split would have been indefensible.



In [ ]:
# === Step 4.5: Derive the near-duplicate structure of the dataset ===
# Uses CLUSTER_LABELS saved in Step 4, so this describes exactly the clustering the
# split was built from rather than recomputing it with possibly different settings.
_clustered = train.copy()
_clustered["_cluster"] = CLUSTER_LABELS
_sizes = _clustered.groupby("_cluster").size()

print("=" * 74)
print("1. CLUSTER SIZE DISTRIBUTION")
print("=" * 74)
print(f"  {len(_clustered)} questions grouped into {len(_sizes)} clusters "
      f"(cosine >= {DEDUP_SIM_THRESHOLD})\n")
print("  cluster size | how many clusters | questions covered")
print("  " + "-" * 54)
for _size, _count in _sizes.value_counts().sort_index().items():
    print(f"  {_size:12d} | {_count:17d} | {_size * _count:17d}")
_present = sorted(_sizes.value_counts().index)
_absent = [n for n in range(1, max(_present) + 1) if n not in _present]
print(f"\n  sizes observed: {_present}")
print(f"  sizes in that range that never occur: {_absent}")

print("\n  grouped into bands:")
for _lo, _hi, _label in [(1, 1, "unique"), (2, 4, "small (2-4)"),
                         (5, 9, "medium (5-9)"), (10, int(_sizes.max()), "large (10+)")]:
    _mask = (_sizes >= _lo) & (_sizes <= _hi)
    _q = int(_sizes[_mask].sum())
    print(f"    {_label:14s}: {int(_mask.sum()):3d} clusters covering {_q:4d} questions "
          f"({_q / len(_clustered):.1%})")

print("\n" + "=" * 74)
print("2. TWO DIFFERENT COUNTS THAT ARE EASY TO CONFUSE")
print("=" * 74)
_n_unique = int((_sizes == 1).sum())
_rows_unique = int(_sizes[_sizes == 1].sum())
_rows_dup = int(_sizes[_sizes > 1].sum())
# Careful: "distinct questions" counts CLUSTERS, "appears once" counts ROWS.
# For the singletons the two coincide, which is exactly why they get mixed up.
print(f"  DISTINCT QUESTIONS (= number of clusters)     : {len(_sizes)}")
print(f"     of which appear exactly once              : {_n_unique}")
print(f"     of which appear 2 or more times           : {int((_sizes > 1).sum())}")
print(f"     check: {_n_unique} + {int((_sizes > 1).sum())} = {_n_unique + int((_sizes > 1).sum())}")
print()
print(f"  ROWS in train.csv                            : {len(_clustered)}")
print(f"     contributed by the once-only questions    : {_rows_unique:4d}"
      f"  ({_rows_unique / len(_clustered):.1%})")
print(f"     contributed by the repeated questions     : {_rows_dup:4d}"
      f"  ({_rows_dup / len(_clustered):.1%})")
print(f"     check: {_rows_unique} + {_rows_dup} = {_rows_unique + _rows_dup}")
print()
print(f"  largest cluster                              : {int(_sizes.max())} rows for ONE question")
print("  -> most of this dataset is not distinct questions. That is the whole")
print("     reason a random split cannot be trusted here.")

_uniq_ids = _sizes[_sizes == 1].index
_example = _clustered[_clustered["_cluster"] == _uniq_ids[3]].iloc[0]
print(f"\n  an example of a genuinely unique question:")
print(f"    id {_example['id']}: {_example['prompt'][:90]}")

print("\n" + "=" * 74)
print("3. EFFECTIVE DATASET SIZE - what we actually have to learn from")
print("=" * 74)
print(f"  rows in train.csv                     : {len(_clustered)}")
print(f"  distinct questions behind those rows  : {len(_sizes)}   (= the cluster count)")
print(f"  average copies per distinct question  : {len(_clustered) / len(_sizes):.2f}")
print("  -> the usable size of this dataset is closer to 613 than to 2,000. Row")
print("     count overstates it by more than 3x, which is worth remembering when")
print("     reading how quickly the from-scratch models reach near-zero train loss.")

# _valid_clusters is the set Step 4 assigned to validation, so the two sides can be
# counted directly instead of matching rows back by content.
_is_valid = _clustered["_cluster"].isin(_valid_clusters)
for _name, _rows, _n_clusters in (
    ("train_split", int((~_is_valid).sum()), _clustered.loc[~_is_valid, "_cluster"].nunique()),
    ("valid_split", int(_is_valid.sum()), _clustered.loc[_is_valid, "_cluster"].nunique()),
):
    print(f"\n  {_name}: {_rows} rows from {_n_clusters} distinct questions "
          f"(avg {_rows / _n_clusters:.2f} copies)")
print(f"\n  distinct questions appearing on BOTH sides: "
      f"{len(set(_clustered.loc[_is_valid, '_cluster']) & set(_clustered.loc[~_is_valid, '_cluster']))}"
      f"  (must be 0 - that is what the split guarantees)")
print("\n  So training sees far fewer distinct questions than 1,600 suggests, and")
print("  validation is scored on distinct questions none of which appear in training.")

print("\n" + "=" * 74)
print("4. THE LARGEST CLUSTER, PRINTED SO THE GROUPING CAN BE CHECKED BY EYE")
print("=" * 74)
_biggest = _clustered[_clustered["_cluster"] == _sizes.idxmax()]
for _, _row in _biggest.head(8).iterrows():
    print(f"  id {_row['id']:4d}  answer {_row['answer']}  {_row['prompt'][:88]}")
if len(_biggest) > 8:
    print(f"  ... and {len(_biggest) - 8} more questions in this same cluster")
print()
print(f"  distinct prompts in this cluster : {_biggest['prompt'].nunique()} of {len(_biggest)} rows")
print(f"  answers in this cluster          : {_biggest['answer'].value_counts().to_dict()}")
print("  -> one underlying question, re-wrapped in different instruction phrasings,")
print("     every copy carrying the same answer. Split these across train and")
print("     validation and the model is scored on questions it was taught.")

del _clustered





## Step 4.6 - Where the artifact baseline's number comes from

Step 4 reported a MAP@3 for a rule that never reads the question. A number like that
should not be taken on trust, so this step derives it in four checkable stages:

1. **The premise.** Show that correct answers and distractors come from disjoint
   pools, by counting how many correct-answer texts are ever reused as a wrong option.
2. **The consequence.** Show that at most one option per question can therefore carry
   the "seen as an answer" fingerprint, which is what makes the rule work.
3. **A worked example.** Print one validation question with each option flagged.
4. **The arithmetic.** Break the score into how many questions the rule placed at
   rank 1, 2, 3 or outside the top 3, and show those counts producing the MAP@3.

Nothing here is recomputed differently from Step 4 - the same `_known_answer_texts`
set and the same `_artifact_rank` function are reused, so the number this step
derives is by construction the number the scoreboard holds.


In [ ]:
# === Step 4.6: Derive the artifact baseline's score step by step ===
# Reuses _known_answer_texts and _artifact_rank from Step 4 so this is provably a
# breakdown of the same computation, not a parallel one that might disagree.
from collections import Counter

print("=" * 74)
print("1. PREMISE: are correct answers and distractors drawn from disjoint pools?")
print("=" * 74)
_all_correct = {row[row["answer"]] for _, row in train.iterrows()}
_all_distractors = {row[c] for _, row in train.iterrows()
                    for c in option_cols if c != row["answer"]}
_overlap = _all_correct & _all_distractors
print(f"  distinct texts used as a CORRECT answer  : {len(_all_correct)}")
print(f"  distinct texts used as a WRONG option    : {len(_all_distractors)}")
print(f"  texts appearing in BOTH roles            : {len(_overlap)}")
print("  -> the pools are disjoint, so a text that was ever a correct answer is")
print("     never a distractor. Recognising the text is enough to identify it.")

print("\n" + "=" * 74)
print("2. CONSEQUENCE: how many options per question carry the fingerprint?")
print("=" * 74)
_n_flagged = Counter(sum(row[c] in _known_answer_texts for c in option_cols)
                     for _, row in valid_split.iterrows())
for _k in sorted(_n_flagged):
    print(f"  {_k} option(s) flagged : {_n_flagged[_k]:3d} validation questions")
print("  -> never 2 or more, which follows from step 1: only the true answer can")
print("     carry a text that the training split used as a correct answer.")
print("     When exactly 1 is flagged, that option is correct with certainty.")
print("     When 0 are flagged the rule is blind and falls back to letter frequency.")

print("\n" + "=" * 74)
print("3. WORKED EXAMPLE: one validation question, each option flagged")
print("=" * 74)
_ex = valid_split.iloc[7]
print(f"  PROMPT: {_ex['prompt'][:96]}")
for _c in option_cols:
    _flag = "FLAGGED (seen as a correct answer in train)" if _ex[_c] in _known_answer_texts else "not seen"
    print(f"    {_c}) [{_flag}]  {str(_ex[_c])[:52]}...")
print(f"  true answer: {_ex['answer']}   rule predicts: {' '.join(_artifact_rank(_ex)[:3])}")

print("\n" + "=" * 74)
print("4. ARITHMETIC: where the true answer landed, and the resulting MAP@3")
print("=" * 74)
_credit = {1: 1.0, 2: 0.5, 3: 1.0 / 3, 0: 0.0}
_ranks = Counter()
for _, _row in valid_split.iterrows():
    _pos = _artifact_rank(_row).index(_row["answer"]) + 1
    _ranks[_pos if _pos <= 3 else 0] += 1

_total = 0.0
for _pos in (1, 2, 3, 0):
    _label = {1: "rank 1", 2: "rank 2", 3: "rank 3", 0: "not in top 3"}[_pos]
    _contrib = _ranks[_pos] * _credit[_pos]
    _total += _contrib
    print(f"  {_label:13s}: {_ranks[_pos]:3d} questions x {_credit[_pos]:.4f} = {_contrib:8.4f}")
print(f"  {'':13s}  {'-' * 34}")
print(f"  {'total credit':13s}: {_total:.4f}  over {len(valid_split)} questions")
print(f"\n  MAP@3 = {_total:.4f} / {len(valid_split)} = {_total / len(valid_split):.4f}")
print(f"  scoreboard holds       {scoreboard[ARTIFACT_BASELINE]:.4f}   (match: "
      f"{abs(_total / len(valid_split) - scoreboard[ARTIFACT_BASELINE]) < 1e-9})")

print("\n  Reading the split: the questions with a flagged option are answered")
print("  correctly with certainty. The rest are guessed by letter frequency, which")
print("  is why some still land at rank 1 by luck and many fall outside the top 3.")


## Step 5 - First real models: TF-IDF + Logistic Regression, TF-IDF + SVM

This is our first method that actually looks at the question text. Each
(prompt, option) pair is turned into one TF-IDF row, labeled 1 if that option
is the correct answer and 0 otherwise, then we train two classic ML
classifiers on those rows: **Logistic Regression** and a **linear SVM**. At
prediction time each of the 5 options for a question gets a score from the
classifier, we rank them, and keep the top 3.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

option_cols = ["A", "B", "C", "D", "E"]

# 1) Fit the vectorizer on the TRAIN SPLIT text only (prompts + options) -> no leakage.
corpus = train_split["prompt"].astype(str).tolist()
for c in option_cols:
    corpus += train_split[c].astype(str).tolist()

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
vectorizer.fit(corpus)
print("Vocabulary size:", len(vectorizer.vocabulary_))


def build_pairwise_rows(df):
    """Turn each question into 5 (prompt+option) rows, one per option,
    labeled 1 if that option is the correct answer, else 0."""
    df = df.reset_index(drop=True)
    texts, labels, qids = [], [], []
    for i, row in df.iterrows():
        for c in option_cols:
            texts.append(str(row["prompt"]) + " [SEP] " + str(row[c]))
            labels.append(1 if row.get("answer") == c else 0)
            qids.append(i)
    return texts, labels, qids

from collections import defaultdict

def scores_to_top3(scores, qids):
    """Group per-option scores back into per-question rankings of top-3 letters."""
    by_q = defaultdict(list)
    for idx, (score, qid) in enumerate(zip(scores, qids)):
        c = option_cols[idx % len(option_cols)]
        by_q[qid].append((c, score))
    preds = []
    for qid in sorted(by_q):
        ranked = sorted(by_q[qid], key=lambda x: x[1], reverse=True)
        preds.append([c for c, _ in ranked[:3]])
    return preds

# 2) Build pairwise train/valid rows and vectorize.
train_texts, train_labels, _ = build_pairwise_rows(train_split)
X_train, y_train = vectorizer.transform(train_texts), train_labels

valid_texts, _, valid_qids = build_pairwise_rows(valid_split)
X_valid = vectorizer.transform(valid_texts)

# 3) TF-IDF + Logistic Regression
logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_valid)[:, 1]
valid_preds_logreg = scores_to_top3(logreg_scores, valid_qids)
scoreboard["TF-IDF + Logistic Regression"] = map_at_3(valid_true, valid_preds_logreg)
print(f"TF-IDF + Logistic Regression MAP@3: {scoreboard['TF-IDF + Logistic Regression']:.4f}")
wandb.log({"TF-IDF + Logistic Regression": scoreboard["TF-IDF + Logistic Regression"]})

# 4) TF-IDF + SVM (LinearSVC - fast and effective on sparse TF-IDF features)
svm = LinearSVC(class_weight="balanced")
svm.fit(X_train, y_train)
svm_scores = svm.decision_function(X_valid)
valid_preds_svm = scores_to_top3(svm_scores, valid_qids)
scoreboard["TF-IDF + SVM"] = map_at_3(valid_true, valid_preds_svm)
print(f"TF-IDF + SVM MAP@3: {scoreboard['TF-IDF + SVM']:.4f}")
wandb.log({"TF-IDF + SVM": scoreboard["TF-IDF + SVM"]})

print("\nScoreboard so far:")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")


## Step 5.5 - (Optional) Early checkpoint submission

You don't have to wait until BERT is done to submit. This cell takes whichever
TF-IDF model scored higher above, **retrains it on the full labeled `train.csv`**
(all 2,000 questions - no need to hold out validation rows for the real
submission), predicts on `test.csv`, and writes `submission.csv` in the exact
format the competition expects. Run Steps 1-5 + this cell, then submit that
file, to sanity-check your whole pipeline before spending time on the bigger
models.

In [ ]:
# === Step 5.5: Early checkpoint submission (TF-IDF baseline) ===
# Pick whichever of the two TF-IDF models did better on the validation split.
best_tfidf_name = max(
    ["TF-IDF + Logistic Regression", "TF-IDF + SVM"],
    key=lambda name: scoreboard[name],
)
print(f"Using '{best_tfidf_name}' for the early submission "
      f"(val MAP@3 = {scoreboard[best_tfidf_name]:.4f})")

# Refit the vectorizer + chosen classifier on ALL labeled data (train, not train_split) -
# more training data almost always helps, and we no longer need a holdout for this.
full_corpus = train["prompt"].astype(str).tolist()
for c in option_cols:
    full_corpus += train[c].astype(str).tolist()

final_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
final_vectorizer.fit(full_corpus)

full_texts, full_labels, _ = build_pairwise_rows(train)
X_full = final_vectorizer.transform(full_texts)

if best_tfidf_name == "TF-IDF + Logistic Regression":
    final_clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    final_clf.fit(X_full, full_labels)
    score_fn = lambda X: final_clf.predict_proba(X)[:, 1]
else:
    final_clf = LinearSVC(class_weight="balanced")
    final_clf.fit(X_full, full_labels)
    score_fn = lambda X: final_clf.decision_function(X)

# Predict on the test set (no "answer" column, so labels are all 0 - ignored).
test_texts, _, test_qids = build_pairwise_rows(test.assign(answer=None))
X_test = final_vectorizer.transform(test_texts)
test_scores = score_fn(X_test)
test_top3 = scores_to_top3(test_scores, test_qids)

checkpoint = pd.DataFrame({
    "ID": test["id"].values,
    "Prediction": [" ".join(row) for row in test_top3],
})
checkpoint.to_csv("submission_checkpoint_tfidf.csv", index=False)

print("Wrote submission_checkpoint_tfidf.csv  | shape:", checkpoint.shape)
print(checkpoint.head(10).to_string(index=False))
print("\nThis is a CHECKPOINT file only, not the real submission - the notebook")
print("picks the single best-scoring model at the very end (Step 12) and writes")
print("THAT to submission.csv. Don'''t submit this file directly.")


## Step 6 - Meaning-based ranking with sentence embeddings (first deep learning)

TF-IDF only counted shared words. Now we use a pretrained transformer - a small, fast one called **MiniLM** - that maps any sentence to a vector capturing its **meaning**, learned from huge amounts of text. Sentences with similar meaning land close together even when they use different words.

Plan:
1. Load the pretrained model (downloads about 80 MB the first time, then it is cached).
2. Encode the prompt and each option into 384-dimensional vectors on the GPU.
3. Rank the options by cosine similarity to the prompt and keep the top 3.

This is the same shape as TF-IDF (vectors -> cosine -> rank), but the vectors now come from a deep neural network instead of word counts.

**Honest expectation:** meaning-similarity is still an imperfect signal here, because every distractor is on-topic and close in meaning to the correct answer. So the gain over the 0.42 baseline may be small - and that is fine. The real jump comes in the next step, when we *train* a model specifically to pick the correct option.



In [ ]:
# === Step 6: Sentence-embedding ranker (MiniLM) ===
from sentence_transformers import SentenceTransformer

# Load a small, fast pretrained embedding model onto our device (GPU if available).
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("Loaded embedding model on:", device)


def embed_rank(df, batch_size=64):
    """Rank the 5 options of each row by cosine similarity of their meaning to the prompt."""
    # Encode into normalized vectors so a dot product equals cosine similarity.
    prompt_emb = embed_model.encode(
        df["prompt"].astype(str).tolist(),
        batch_size=batch_size, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    option_emb = {
        c: embed_model.encode(
            df[c].astype(str).tolist(),
            batch_size=batch_size, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=False,
        )
        for c in option_cols
    }
    preds = []
    for i in range(len(df)):
        sims = [(c, float(prompt_emb[i] @ option_emb[c][i])) for c in option_cols]
        sims.sort(key=lambda x: x[1], reverse=True)
        preds.append([c for c, _ in sims[:3]])
    return preds


valid_preds_emb = embed_rank(valid_split)
scoreboard["Sentence embeddings (MiniLM)"] = map_at_3(valid_true, valid_preds_emb)

print(f"\nMiniLM embedding MAP@3 on validation: {scoreboard['Sentence embeddings (MiniLM)']:.4f}\n")
print("Scoreboard so far:")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")

wandb.log({"Sentence embeddings (MiniLM)": scoreboard["Sentence embeddings (MiniLM)"]})

Conclusion:

On the leakage-free split, TF-IDF + Logistic Regression reaches 0.7733 and
TF-IDF + SVM 0.7596, while zero-shot MiniLM manages only 0.4846 - barely above
the 0.4088 constant baseline.

Two things are worth reading off this. First, supervised lexical overlap is a
genuinely strong signal here: a bag-of-words model with no notion of meaning
gets most of the way. Second, meaning-similarity on its own is close to
worthless for this task, and the reason is visible in the data - every
distractor is on-topic and phrased like the correct answer, so cosine
similarity to the prompt cannot separate them. That is exactly the gap
fine-tuning closes in Step 8.

An earlier version of this notebook reported 0.9721 for TF-IDF + SVM. That
number came from a random split, where near-duplicate questions appeared on
both sides and the classifier was partly recalling rows it had already seen.
The figures above use the cluster split from Step 4 and are the honest ones.


## Step 6.5 - A from-scratch model (our own architecture, trained from random init)

This is the **"train a model from scratch"** requirement: we write the `nn.Module`
ourselves (embedding layer + BiLSTM encoder + a small scoring head) and every
weight starts randomly initialized -- nothing pretrained is loaded here. Compare
this to Step 6 (MiniLM), which is a *pretrained* model used as-is, and Step 8
(BERT), which *fine-tunes* pretrained weights. This one learns everything,
including the word embeddings, only from our own 2,000 training questions.

In [ ]:
# === Step 6.5: From-scratch model - a custom BiLSTM classifier ===
# Own architecture, own vocabulary, randomly-initialized weights, trained only
# on train_split. No pretrained checkpoints are loaded anywhere in this cell.
import re
import torch
import torch.nn as nn
from collections import Counter

torch.manual_seed(42)


def simple_tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


# 1) Build our own vocabulary from the train split only (no leakage).
counter = Counter()
for col in ["prompt"] + option_cols:
    for text in train_split[col].astype(str):
        counter.update(simple_tokenize(text))

MIN_FREQ = 2
scratch_vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        scratch_vocab[word] = len(scratch_vocab)
VOCAB_SIZE = len(scratch_vocab)
print("From-scratch model vocabulary size:", VOCAB_SIZE)

MAX_LEN_SCRATCH = 40


def encode(text, max_len=MAX_LEN_SCRATCH):
    ids = [scratch_vocab.get(tok, 1) for tok in simple_tokenize(text)[:max_len]]
    ids = ids + [0] * (max_len - len(ids))
    return ids


class ScratchMCQDataset(torch.utils.data.Dataset):
    """Same (prompt, 5 options, label) shape as the BERT dataset below, but
    tokenized with our own vocabulary instead of a pretrained tokenizer."""

    def __init__(self, df, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.has_labels = has_labels and ("answer" in df.columns)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        prompt_ids = torch.tensor(encode(row["prompt"]), dtype=torch.long)
        option_ids = torch.stack(
            [torch.tensor(encode(row[c]), dtype=torch.long) for c in option_cols]
        )
        item = {"prompt_ids": prompt_ids, "option_ids": option_ids}
        if self.has_labels:
            item["label"] = torch.tensor(letter_to_idx_scratch[row["answer"]], dtype=torch.long)
        return item


letter_to_idx_scratch = {l: i for i, l in enumerate(option_cols)}


class ScratchEncoder(nn.Module):
    """Trainable embedding + bidirectional LSTM, mean-pooled over time."""

    def __init__(self, vocab_size, emb_dim=100, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)  # random init
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, ids):
        mask = (ids != 0).unsqueeze(-1).float()
        emb = self.embedding(ids)
        out, _ = self.lstm(emb)
        out = out * mask
        summed = out.sum(1)
        counts = mask.sum(1).clamp(min=1)
        return summed / counts  # (B, hidden_dim*2)


class ScratchMCQModel(nn.Module):
    """Our own multiple-choice architecture: encode prompt + each option with
    the SAME encoder, combine with a small MLP, output one logit per option."""

    def __init__(self, vocab_size, emb_dim=100, hidden_dim=128):
        super().__init__()
        self.encoder = ScratchEncoder(vocab_size, emb_dim, hidden_dim)
        feat_dim = hidden_dim * 2
        self.scorer = nn.Sequential(
            nn.Linear(feat_dim * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, prompt_ids, option_ids):
        B, K, L = option_ids.shape
        p_vec = self.encoder(prompt_ids)                                  # (B, feat)
        o_vec = self.encoder(option_ids.view(B * K, L)).view(B, K, -1)    # (B, K, feat)
        p_exp = p_vec.unsqueeze(1).expand(-1, K, -1)
        feats = torch.cat([p_exp, o_vec, (p_exp - o_vec).abs(), p_exp * o_vec], dim=-1)
        return self.scorer(feats).squeeze(-1)                             # (B, K)


scratch_train_ds = ScratchMCQDataset(train_split)
scratch_valid_ds = ScratchMCQDataset(valid_split)
scratch_train_loader = torch.utils.data.DataLoader(scratch_train_ds, batch_size=32, shuffle=True)
scratch_valid_loader = torch.utils.data.DataLoader(scratch_valid_ds, batch_size=64, shuffle=False)

scratch_model = ScratchMCQModel(VOCAB_SIZE).to(device)
scratch_optimizer = torch.optim.Adam(scratch_model.parameters(), lr=1e-3)
scratch_criterion = nn.CrossEntropyLoss()

SCRATCH_EPOCHS = 8
print(f"Training the from-scratch BiLSTM model for {SCRATCH_EPOCHS} epochs on {device} ...")

for epoch in range(1, SCRATCH_EPOCHS + 1):
    scratch_model.train()
    running = 0.0
    for batch in scratch_train_loader:
        labels = batch["label"].to(device)
        prompt_ids = batch["prompt_ids"].to(device)
        option_ids = batch["option_ids"].to(device)
        scratch_optimizer.zero_grad()
        logits = scratch_model(prompt_ids, option_ids)
        loss = scratch_criterion(logits, labels)
        loss.backward()
        scratch_optimizer.step()
        running += loss.item()
    # Evaluate every epoch so W&B gets a real training curve for this model too,
    # matching the per-epoch logging used for BERT and the Step 6.7 encoders.
    scratch_model.eval()
    _epoch_preds = []
    with torch.no_grad():
        for batch in scratch_valid_loader:
            _lg = scratch_model(batch["prompt_ids"].to(device),
                                batch["option_ids"].to(device)).cpu().numpy()
            _epoch_preds.extend([[option_cols[j] for j in row[:3]] for row in (-_lg).argsort(axis=1)])
    _epoch_val = map_at_3(valid_true, _epoch_preds)
    _epoch_loss = running / len(scratch_train_loader)
    print(f"  epoch {epoch}/{SCRATCH_EPOCHS}  avg train loss {_epoch_loss:.4f}  val MAP@3 {_epoch_val:.4f}")
    wandb.log({"bilstm_scratch/train_loss": _epoch_loss,
               "bilstm_scratch/val_map_at_3": _epoch_val,
               "scratch_epoch": epoch})
    if epoch == 1:   # reset on re-run so curves never accumulate across executions
        TRAINING_HISTORY["From-scratch BiLSTM"] = {"epoch": [], "train_loss": [], "val_map_at_3": []}
    _hist = TRAINING_HISTORY["From-scratch BiLSTM"]
    _hist["epoch"].append(epoch)
    _hist["train_loss"].append(_epoch_loss)
    _hist["val_map_at_3"].append(_epoch_val)

show_training_curve("From-scratch BiLSTM", "fig_curve_bilstm.png")

# Evaluate with the same MAP@3 metric used everywhere else.
scratch_model.eval()
preds_scratch = []
with torch.no_grad():
    for batch in scratch_valid_loader:
        prompt_ids = batch["prompt_ids"].to(device)
        option_ids = batch["option_ids"].to(device)
        logits = scratch_model(prompt_ids, option_ids).cpu().numpy()
        ranked = (-logits).argsort(axis=1)
        preds_scratch.extend([[option_cols[j] for j in row[:3]] for row in ranked])

scoreboard["From-scratch BiLSTM"] = map_at_3(valid_true, preds_scratch)
print(f"\nFrom-scratch BiLSTM MAP@3 on validation: {scoreboard['From-scratch BiLSTM']:.4f}\n")
print("Scoreboard so far:")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")

wandb.log({"From-scratch BiLSTM": scoreboard["From-scratch BiLSTM"]})





## Step 6.6 - (Optional) Submission checkpoint: from-scratch BiLSTM

Same idea as Step 5.5, now for the from-scratch model. We retrain
`ScratchMCQModel` on the **full** `train.csv` (2,000 questions instead of just
`train_split`'s ~1,600), predict on `test.csv`, and overwrite
`submission.csv`. A couple of notes before you submit this one:

- Train loss drops from 1.2922 to about 0.04 by epoch 4 and then flattens for
  the remaining epochs - the model has fitted the training set to near-zero
  loss. With only ~1,600 training rows and a vocabulary of a few thousand
  tokens, that much memorisation is expected. Validation MAP@3 is 0.8633,
  which is a genuinely held-out number on the leakage-free split.
- That 0.8633 is the striking result of this notebook. A BiLSTM built from
  random initialisation, with a 2,777-word vocabulary learned from 1,600
  questions, lands within 0.002 of a fine-tuned bert-base-uncased (0.8650).
  Watch the per-epoch curve rather than the final number, though: validation
  peaks at 0.8942 on epoch 7 and falls back to 0.8633 on epoch 8, so the run
  is noisy and the gap to BERT is well inside that noise.
- Training on the full 2,000 rows below uses the same 8 epochs. If you see
  train loss bottom out again after 3-4 epochs, that's fine - it just means
  it's converged, not that something's broken.




In [ ]:
# === Step 6.6: Submission checkpoint (from-scratch BiLSTM) ===
# Rebuild the vocabulary from ALL labeled data (train, not train_split) -
# more data helps, and we no longer need a holdout for this run.
full_counter = Counter()
for col in ["prompt"] + option_cols:
    for text in train[col].astype(str):
        full_counter.update(simple_tokenize(text))

final_scratch_vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in full_counter.items():
    if freq >= MIN_FREQ:
        final_scratch_vocab[word] = len(final_scratch_vocab)
print("Final from-scratch vocabulary size:", len(final_scratch_vocab))


def final_encode(text, max_len=MAX_LEN_SCRATCH):
    ids = [final_scratch_vocab.get(tok, 1) for tok in simple_tokenize(text)[:max_len]]
    return ids + [0] * (max_len - len(ids))


class FinalScratchDataset(torch.utils.data.Dataset):
    def __init__(self, df, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.has_labels = has_labels and ("answer" in df.columns)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        prompt_ids = torch.tensor(final_encode(row["prompt"]), dtype=torch.long)
        option_ids = torch.stack(
            [torch.tensor(final_encode(row[c]), dtype=torch.long) for c in option_cols]
        )
        item = {"prompt_ids": prompt_ids, "option_ids": option_ids}
        if self.has_labels:
            item["label"] = torch.tensor(letter_to_idx_scratch[row["answer"]], dtype=torch.long)
        return item


final_train_ds = FinalScratchDataset(train)
final_train_loader = torch.utils.data.DataLoader(final_train_ds, batch_size=32, shuffle=True)

final_scratch_model = ScratchMCQModel(len(final_scratch_vocab)).to(device)
final_optimizer = torch.optim.Adam(final_scratch_model.parameters(), lr=1e-3)

print(f"Retraining the from-scratch BiLSTM on all {len(train)} questions for {SCRATCH_EPOCHS} epochs ...")
for epoch in range(1, SCRATCH_EPOCHS + 1):
    final_scratch_model.train()
    running = 0.0
    for batch in final_train_loader:
        labels = batch["label"].to(device)
        prompt_ids = batch["prompt_ids"].to(device)
        option_ids = batch["option_ids"].to(device)
        final_optimizer.zero_grad()
        logits = final_scratch_model(prompt_ids, option_ids)
        loss = scratch_criterion(logits, labels)
        loss.backward()
        final_optimizer.step()
        running += loss.item()
    print(f"  epoch {epoch}/{SCRATCH_EPOCHS}  avg train loss {running / len(final_train_loader):.4f}")

# Predict on the test set (no "answer" column).
final_test_ds = FinalScratchDataset(test, has_labels=False)
final_test_loader = torch.utils.data.DataLoader(final_test_ds, batch_size=64, shuffle=False)

final_scratch_model.eval()
test_top3_scratch = []
with torch.no_grad():
    for batch in final_test_loader:
        prompt_ids = batch["prompt_ids"].to(device)
        option_ids = batch["option_ids"].to(device)
        logits = final_scratch_model(prompt_ids, option_ids).cpu().numpy()
        ranked = (-logits).argsort(axis=1)
        test_top3_scratch.extend([[option_cols[j] for j in row[:3]] for row in ranked])

checkpoint = pd.DataFrame({
    "ID": test["id"].values,
    "Prediction": [" ".join(row) for row in test_top3_scratch],
})
checkpoint.to_csv("submission_checkpoint_scratch.csv", index=False)

print("Wrote submission_checkpoint_scratch.csv  | shape:", checkpoint.shape)
print(checkpoint.head(10).to_string(index=False))
print("\nThis is a CHECKPOINT file only, not the real submission - the notebook")
print("picks the single best-scoring model at the very end (Step 12) and writes")
print("THAT to submission.csv. Don'''t submit this file directly.")


## Step 6.7 - Two more from-scratch encoders: text-CNN and a mini transformer

Step 6.5 built a BiLSTM from random initialisation. This step adds two more
encoders trained the same way, so that three different architectures can be
compared while holding everything else constant - same vocabulary, same
`[p; o; |p-o|; p*o]` scoring head, same Adam optimiser, same number of epochs.
The only thing that changes is how the sequence is encoded:

| encoder | mechanism | processes the sequence |
|---|---|---|
| BiLSTM (Step 6.5) | recurrence with gating | in order, both directions |
| text-CNN | convolutions of width 3/4/5 + max-over-time | in parallel, local n-grams |
| mini transformer | multi-head self-attention, 2 blocks | in parallel, every token sees every other |

The mini transformer is the interesting one. It uses the **same mechanism as BERT**
but with **no pretraining at all**. Comparing it against the fine-tuned BERT of
Step 8 therefore separates two effects that are otherwise tangled together:

- *architecture*: mini transformer vs BiLSTM, both untrained -> value of self-attention
- *pretraining*: fine-tuned BERT vs mini transformer, both transformers -> value of pretraining

Each model logs a **per-epoch curve** to Weights and Biases (`train_loss` and
`val_map_at_3`), so the W&B workspace shows real training curves for these models
rather than a single end-of-run number.


In [ ]:
# === Step 6.7: Two more from-scratch encoders - text-CNN and a mini transformer ===
# Everything here is trained from RANDOM initialisation: own vocabulary, own
# embeddings, no pretrained weights anywhere.
#
# All three from-scratch models share the SAME scoring head, vocabulary, optimiser
# and epoch count as Step 6.5, so the only variable is the encoder:
#   BiLSTM           - recurrent, reads the sequence in order      (Step 6.5)
#   text-CNN         - convolutional, parallel n-gram detectors    (here)
#   mini transformer - self-attention, same family as BERT but NOT pretrained (here)
#
# The mini transformer is the useful control: comparing it against the fine-tuned
# BERT of Step 8 isolates the value of PRETRAINING while holding the architecture
# family fixed, which the zero-shot MiniLM comparison cannot do.
import torch.nn as nn


class CNNEncoder(nn.Module):
    """Text-CNN. Convolutions of width 3/4/5 act as parallel n-gram detectors and
    max-over-time pooling keeps the strongest activation of each filter. No
    recurrence, so the whole sequence is processed in parallel."""

    def __init__(self, vocab_size, emb_dim=100, n_filters=100, kernel_sizes=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList(
            [nn.Conv1d(emb_dim, n_filters, k, padding=k // 2) for k in kernel_sizes]
        )
        self.dropout = nn.Dropout(0.2)
        self.out_dim = n_filters * len(kernel_sizes)

    def forward(self, ids):
        seq_len = ids.size(1)
        mask = (ids != 0).unsqueeze(1).float()
        x = self.embedding(ids).transpose(1, 2)          # (B, emb, L)
        feats = []
        for conv in self.convs:
            h = torch.relu(conv(x))[..., :seq_len] * mask
            feats.append(h.max(dim=2).values)            # max-over-time
        return self.dropout(torch.cat(feats, dim=-1))


class TransformerBlock(nn.Module):
    """One transformer block written out explicitly:
    self-attention -> residual + LayerNorm -> feed-forward -> residual + LayerNorm."""

    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(),
                                nn.Linear(ff_dim, d_model))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, pad_mask):
        attended, _ = self.attn(x, x, x, key_padding_mask=pad_mask, need_weights=False)
        x = self.norm1(x + self.dropout(attended))
        return self.norm2(x + self.dropout(self.ff(x)))


class TinyTransformerEncoder(nn.Module):
    """Two transformer blocks with learned positional embeddings. Same mechanism as
    BERT (multi-head self-attention) but every weight starts random."""

    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, ff_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.positions = nn.Embedding(MAX_LEN_SCRATCH, d_model)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, ff_dim) for _ in range(n_layers)]
        )
        self.out_dim = d_model

    def forward(self, ids):
        pad_mask = ids == 0
        pos = torch.arange(ids.size(1), device=ids.device).unsqueeze(0)
        x = self.embedding(ids) + self.positions(pos)
        for block in self.blocks:
            x = block(x, pad_mask)
        mask = (~pad_mask).unsqueeze(-1).float()
        return (x * mask).sum(1) / mask.sum(1).clamp(min=1)


class PairScorer(nn.Module):
    """The Step 6.5 scoring head, reused verbatim so encoders stay comparable:
    concat [prompt, option, |prompt - option|, prompt * option] -> MLP -> 1 logit."""

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        feat_dim = encoder.out_dim
        self.scorer = nn.Sequential(
            nn.Linear(feat_dim * 4, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1)
        )

    def forward(self, prompt_ids, option_ids):
        B, K, L = option_ids.shape
        p_vec = self.encoder(prompt_ids)
        o_vec = self.encoder(option_ids.view(B * K, L)).view(B, K, -1)
        p_exp = p_vec.unsqueeze(1).expand(-1, K, -1)
        feats = torch.cat([p_exp, o_vec, (p_exp - o_vec).abs(), p_exp * o_vec], dim=-1)
        return self.scorer(feats).squeeze(-1)


def train_scratch_encoder(label, wandb_tag, encoder):
    """Train one encoder and log a per-epoch curve to W&B so the run has real
    line charts (train loss and validation MAP@3), not just a single number."""
    torch.manual_seed(42)
    model = PairScorer(encoder).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n{label}: {n_params:,} parameters, {SCRATCH_EPOCHS} epochs on {device}")

    for epoch in range(1, SCRATCH_EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in scratch_train_loader:
            optimizer.zero_grad()
            logits = model(batch["prompt_ids"].to(device), batch["option_ids"].to(device))
            loss = criterion(logits, batch["label"].to(device))
            loss.backward()
            optimizer.step()
            running += loss.item()
        train_loss = running / len(scratch_train_loader)

        model.eval()
        preds = []
        with torch.no_grad():
            for batch in scratch_valid_loader:
                lg = model(batch["prompt_ids"].to(device),
                           batch["option_ids"].to(device)).cpu().numpy()
                preds.extend([[option_cols[j] for j in row[:3]] for row in (-lg).argsort(axis=1)])
        val = map_at_3(valid_true, preds)
        print(f"  epoch {epoch}/{SCRATCH_EPOCHS}  train loss {train_loss:.4f}  val MAP@3 {val:.4f}")
        wandb.log({f"{wandb_tag}/train_loss": train_loss,
                   f"{wandb_tag}/val_map_at_3": val,
                   "scratch_epoch": epoch})
        if epoch == 1:
            TRAINING_HISTORY[label] = {"epoch": [], "train_loss": [], "val_map_at_3": []}
        TRAINING_HISTORY[label]["epoch"].append(epoch)
        TRAINING_HISTORY[label]["train_loss"].append(train_loss)
        TRAINING_HISTORY[label]["val_map_at_3"].append(val)

    scoreboard[label] = val
    show_training_curve(label, f"fig_curve_{wandb_tag}.png")
    wandb.summary[f"{wandb_tag}/final_map_at_3"] = val
    wandb.summary[f"{wandb_tag}/parameters"] = n_params
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return val


train_scratch_encoder("From-scratch text-CNN", "cnn_scratch", CNNEncoder(VOCAB_SIZE))
train_scratch_encoder("From-scratch mini transformer", "transformer_scratch",
                      TinyTransformerEncoder(VOCAB_SIZE))

print("\nScoreboard so far:")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")




## Step 7 - Prepare the data for multiple-choice fine-tuning

So far every method used fixed, pretrained knowledge. Now we will **train** a model on our 1,600 labeled questions so it learns from the data how to pick the correct option.

The standard "multiple choice" setup works like this:
- Each question becomes **5 (prompt, option) pairs** - one per choice.
- The model reads each pair and outputs a single **score (logit)** for that option.
- A softmax over the 5 scores turns them into probabilities; the **index of the correct option** is the training target.
- Training adjusts the model so the correct option gets the highest score.

This step only **prepares** the data (training is Step 8). We:
1. Load the tokenizer for `bert-base-uncased` - it converts text into the integer tokens the model expects.
2. Wrap each split in a Dataset that, per question, produces a tensor of shape `(5 choices, sequence length)` plus the correct index.
3. Print one prepared item so the shapes are concrete.

Why `bert-base-uncased`? It is small, rock-solid, and fits comfortably in 4 GB. We can try a stronger model later.

Run the cell below (downloads the tokenizer, a few MB).

In [ ]:
# === Step 7: Prepare data for multiple-choice fine-tuning ===
import torch
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased" #"roberta-base" #"microsoft/deberta-v3-small"
# Measured on the real data: the longest (prompt, option) pair is 200 tokens and0
000000
# p99 is 131, so 256 truncates nothing at all (192 clipped a handful of pairs).
# We do NOT pad to this length - see mcq_collate below.
MAX_LEN = 256
letters = ["A", "B", "C", "D", "E"]
letter_to_idx = {l: i for i, l in enumerate(letters)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loaded tokenizer for:", MODEL_NAME)


class MCQDataset(torch.utils.data.Dataset):
    """One item = a single question turned into 5 (prompt, option) pairs (+ correct index)."""

    def __init__(self, df, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.has_labels = has_labels and ("answer" in df.columns)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        prompt = str(row["prompt"])
        first = [prompt] * 5                                   # same prompt, repeated
        second = [str(row[l]) for l in letters]               # the 5 options
        # No padding here - mcq_collate pads each batch to its own longest sequence.
        enc = tokenizer(first, second, truncation=True, max_length=MAX_LEN)
        item = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"]
        if self.has_labels:
            item["label"] = letter_to_idx[row["answer"]]
        return item


def mcq_collate(batch):
    """Pad every sequence in this batch to the batch's OWN longest sequence.

    The median (prompt, option) pair is only 56 tokens, so padding everything to
    a fixed MAX_LEN would spend most of the GPU time pushing padding tokens
    through the model. Padding per batch instead cuts that waste substantially,
    which is what buys the extra training epochs the model actually needs.
    Returns tensors shaped (batch, 5, seq_len) - what AutoModelForMultipleChoice wants.
    """
    seq_keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in batch[0]]
    batch_max = max(len(seq) for item in batch for seq in item["input_ids"])
    out = {}
    for key in seq_keys:
        pad_value = tokenizer.pad_token_id if key == "input_ids" else 0
        out[key] = torch.tensor(
            [[seq + [pad_value] * (batch_max - len(seq)) for seq in item[key]] for item in batch],
            dtype=torch.long,
        )
    if "label" in batch[0]:
        out["label"] = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return out


train_ds = MCQDataset(train_split)
valid_ds = MCQDataset(valid_split)
print(f"train_ds: {len(train_ds)} questions   valid_ds: {len(valid_ds)} questions")

# Peek at one prepared item to understand the shapes.
sample = train_ds[0]
print("\nOne question becomes 5 (prompt, option) sequences:")
print("  number of options:", len(sample["input_ids"]))
print("  token lengths    :", [len(x) for x in sample["input_ids"]])
print("  label (index of correct option):", sample["label"], "->", letters[sample["label"]])
print("\n  Decoded choice A (prompt + option joined):")
print("   ", tokenizer.decode(sample["input_ids"][0], skip_special_tokens=True)[:200], "...")

# Show what dynamic padding actually saves on one real batch.
_demo = mcq_collate([train_ds[i] for i in range(min(16, len(train_ds)))])
print(f"\nDynamic padding: a 16-question batch pads to {_demo['input_ids'].shape[-1]} tokens,"
      f" not a fixed {MAX_LEN}.")


## Step 8 - Fine-tune BERT for multiple choice (the real training)

Now we train. We load `bert-base-uncased` with a **multiple-choice head** on top (a tiny classifier that turns each option's reading into one score). That head starts with random weights, so the model is useless until trained - you will see a warning saying exactly that, which is normal and expected.

The loop is the same shape as every deep-learning training loop:
1. Take a batch of questions.
2. **Forward pass** - the model scores the 5 options; cross-entropy loss compares those scores against the correct index.
3. **Backward pass** - gradients say how to nudge each weight to lower the loss.
4. **Optimizer step** - update the weights.
5. Repeat over all batches, for a couple of epochs (passes through the data).

Practical choices for your 4 GB GPU:
- **Mixed precision (fp16)** roughly halves memory and speeds things up.
- **Small batch + gradient accumulation** keeps memory low while the effective batch stays reasonable.
- If you ever hit a CUDA out-of-memory error, set `TRAIN_BATCH = 1` and re-run.
- **LoRA (`USE_LORA = True`)**: instead of updating all ~110M BERT weights, we freeze the
  pretrained weights and train small low-rank adapter matrices injected into the attention
  `query`/`value` projections, plus the (new, randomly-initialized) classifier head. This
  trains a small fraction of the total parameters, uses less GPU memory, and trains faster.
  Set `USE_LORA = False` to go back to full fine-tuning if you want to compare.

Three details worth knowing, because they are all visible in the cell output:

- **Learning-rate schedule**: a linear warmup followed by linear decay
  (`get_linear_schedule_with_warmup`), so the first few hundred steps ease the
  randomly-initialised head in before the pretrained weights are disturbed.
- **Best-epoch tracking**: we keep the epoch with the highest validation MAP@3,
  not simply the last one. Step 10 retrains for that many epochs and Step 12
  reuses the validation logits from that same epoch, so a model's ensemble
  weight and its rankings always come from one consistent checkpoint.
- **Dynamic padding**: each batch is padded to its own longest sequence rather
  than a fixed 256 tokens, which cuts the compute per batch substantially given
  how short these prompts are.

After each epoch we score on validation with our MAP@3 and add it to the
scoreboard. At the end the cell plots the training curve and a confusion matrix
over the option letters.

**Heads-up:** the first run downloads the model (~440 MB), then trains for a few minutes. Watch the loss fall and validation MAP@3 climb - this is where we expect to finally beat 0.42.

Run the cell below.


In [ ]:
# === Step 8: Fine-tune BERT for multiple choice ===
import math
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model

# ---- knobs (adaptive to the device so this doesn't take 11+ hours on CPU) ----
# On CPU each step is ~30-50x slower than on a T4/P100 GPU, so we shrink the
# batch there just to keep things runnable; on GPU we use a much bigger batch,
# which also means far fewer optimizer steps for the same amount of data.
if device == "cuda":
    TRAIN_BATCH = 8        # questions per micro-step (each question = 5 sequences)
    ACCUM_STEPS = 2        # effective batch = 16 questions
    EVAL_BATCH = 32
else:
    TRAIN_BATCH = 2
    ACCUM_STEPS = 4        # effective batch = TRAIN_BATCH * ACCUM_STEPS = 8 questions
    EVAL_BATCH = 8

# Full fine-tuning instead of LoRA. The evidence for the switch is in the earlier
# run's own numbers: with LoRA the validation MAP@3 was still climbing steeply at
# the LAST epoch (0.56 -> 0.73 -> 0.80 -> 0.82) and average train loss was still
# 0.37, i.e. the model never converged - it was under-trained, not maxed out.
# LoRA only nudges a small low-rank subspace, so it needs many more epochs to
# reach where full fine-tuning gets in a few. With only ~1,600 training
# questions, updating all the weights is affordable and lands higher.
USE_LORA = False
EPOCHS = 8 if USE_LORA else 10
LR = 2e-4 if USE_LORA else 2e-5   # LoRA typically wants a higher LR than full fine-tuning
WARMUP_FRAC = 0.1   # linear warmup then linear decay - the standard transformer recipe

torch.manual_seed(42)
use_amp = (device == "cuda")

wandb.config.update({
    "model_name": MODEL_NAME,
    "epochs": EPOCHS,
    "train_batch": TRAIN_BATCH,
    "accum_steps": ACCUM_STEPS,
    "eval_batch": EVAL_BATCH,
    "learning_rate": LR,
    "use_lora": USE_LORA,
    "warmup_frac": WARMUP_FRAC,
    "max_len": MAX_LEN,
})

# Free the Step-6 embedding model from the GPU so we have room to train.
if "embed_model" in globals():
    del embed_model
torch.cuda.empty_cache()

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(device)

if USE_LORA:
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["query", "key", "value", "dense"],   # attention + FFN projections
        modules_to_save=["classifier"],       # the multiple-choice head is new, train it fully
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH, shuffle=True,
                          num_workers=0, collate_fn=mcq_collate)
valid_loader = DataLoader(valid_ds, batch_size=EVAL_BATCH, shuffle=False,
                          num_workers=0, collate_fn=mcq_collate)

# Warm the LR up over the first 10% of optimizer steps, then decay it linearly to 0.
# Fine-tuning a pretrained transformer at a flat LR from step 0 tends to disturb the
# pretrained weights early and leave accuracy on the table.
total_opt_steps = math.ceil(len(train_loader) / ACCUM_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(WARMUP_FRAC * total_opt_steps), total_opt_steps
)


def evaluate():
    """Score the validation set: rank the 5 option-logits, take top 3, compute MAP@3.

    Returns (map_at_3, logits) so the caller can cache the BEST epoch's logits
    rather than whichever epoch happened to run last.
    """
    model.eval()
    logits_all = []
    with torch.no_grad():
        for batch in valid_loader:
            batch.pop("label", None)
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits = model(**batch).logits            # (B, 5)
            logits_all.append(logits.float().cpu().numpy())
    logits_all = np.concatenate(logits_all, axis=0)        # (N, 5)
    ranked = (-logits_all).argsort(axis=1)                 # highest score first
    preds = [[letters[j] for j in row[:3]] for row in ranked]
    return map_at_3(valid_true, preds), logits_all


print(f"Training {MODEL_NAME}: {EPOCHS} epochs, effective batch {TRAIN_BATCH * ACCUM_STEPS}, on {device}")
print("(the random multiple-choice head warning below is expected)\n")

# Track the BEST epoch, not just the last one. Two things depend on this:
# Step 12 needs the validation logits from the same epoch whose score it uses as
# the ensemble weight (otherwise the weight and the rankings disagree), and
# Step 10 should retrain on the full data for the epoch count that actually
# scored best here rather than blindly reusing EPOCHS.
best_val, BEST_EPOCH = -1.0, EPOCHS
# Label follows MODEL_NAME so the scoreboard never claims "BERT" when a
# different backbone (roberta-base, deberta-v3-small, ...) was actually trained.
BACKBONE = MODEL_NAME.split("/")[-1]
model_label = f"{BACKBONE} + LoRA" if USE_LORA else f"{BACKBONE} fine-tuned"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, start=1):
        labels = batch.pop("label").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            out = model(**batch, labels=labels)
            loss = out.loss / ACCUM_STEPS
        scaler.scale(loss).backward()
        running += out.loss.item()
        if step % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        if step % 200 == 0:
            print(f"  epoch {epoch}  step {step:4d}/{len(train_loader)}  running loss {running / step:.4f}")
            wandb.log({"train/loss": running / step, "epoch": epoch, "step": step})
    val, val_logits = evaluate()
    scoreboard[f"{model_label} (after epoch {epoch})"] = val
    if val > best_val:
        best_val, BEST_EPOCH = val, epoch
        bert_valid_logits_final = val_logits   # cache the BEST epoch for Step 12
    avg_train_loss = running / len(train_loader)
    print(f"epoch {epoch} complete | avg train loss {avg_train_loss:.4f} | validation MAP@3 {val:.4f}\n")
    wandb.log({"epoch": epoch, "train/avg_loss": avg_train_loss, "val/map_at_3": val,
               "train/lr": scheduler.get_last_lr()[0]})
    if epoch == 1:
        TRAINING_HISTORY[model_label] = {"epoch": [], "train_loss": [], "val_map_at_3": []}
    TRAINING_HISTORY[model_label]["epoch"].append(epoch)
    TRAINING_HISTORY[model_label]["train_loss"].append(avg_train_loss)
    TRAINING_HISTORY[model_label]["val_map_at_3"].append(val)

show_training_curve(model_label, "fig_curve_finetuned.png")
show_confusion_matrix(bert_valid_logits_final, valid_true, list(option_cols),
                      f"{model_label} on the validation split",
                      "fig_confusion_matrix.png")

print(f"Best epoch: {BEST_EPOCH}/{EPOCHS} with validation MAP@3 {best_val:.4f}")
print("(Step 10 retrains on the full data for that many epochs.)\n")

print("Scoreboard so far:")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")

wandb.log({"scoreboard_final": scoreboard})





## Step 9 - Predict the test set and write submission.csv

This is the deliverable. We reuse the fine-tuned `model`, now on the 500 test questions (which have no `answer` column). Steps:
1. Wrap `test` in our `MCQDataset` with `has_labels=False`.
2. Run the model to get 5 option-scores per question.
3. Rank them and keep the **top 3 letters**, formatted as a single string like `B A C`.
4. Save a CSV with columns `ID` and `Prediction`, exactly matching `sample_submission.csv`.

As a sanity check we also print the distribution of our top-1 predictions: it should be spread across A-E (leaning to B/C, like the training data), not collapsed onto a single letter - that would signal a bug.

Run the cell below.

In [ ]:
# === Step 9: Predict the test set and write submission.csv ===
test_ds = MCQDataset(test, has_labels=False)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH, shuffle=False,
                         num_workers=0, collate_fn=mcq_collate)

model.eval()
logits_all = []
with torch.no_grad():
    for batch in test_loader:
        batch.pop("label", None)                      # the test set has no labels
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits            # (B, 5)
        logits_all.append(logits.float().cpu().numpy())

test_logits = np.concatenate(logits_all, axis=0)      # (500, 5)
ranked = (-test_logits).argsort(axis=1)               # best option first
top3 = [" ".join(letters[j] for j in row[:3]) for row in ranked]

checkpoint = pd.DataFrame({"ID": test["id"].values, "Prediction": top3})
checkpoint.to_csv("submission_checkpoint_bert_initial.csv", index=False)

print("Wrote submission_checkpoint_bert_initial.csv  | shape:", checkpoint.shape)
print(checkpoint.head(10).to_string(index=False))
print("\nThis is a CHECKPOINT file only (train_split-trained model) - Step 10")
print("retrains on all the data, and Step 12 picks the best model overall and")
print("writes THAT to submission.csv. Don'''t submit this file directly.")

# Sanity check: distribution of our most-confident (top-1) predicted letter.
print("\nTop-1 predicted letter distribution on test:")
first_letter = checkpoint["Prediction"].str[0]
print(first_letter.value_counts().reindex(letters).fillna(0).astype(int).to_string())


## Step 10 - Retrain on all 2,000 and rebuild the final submission

In Step 8 we held out 400 questions to *prove* the method works (best validation MAP@3 0.8650, at epoch 6 of 10). For the **final** model we no longer need that holdout - more training data almost always helps - so we retrain the same BERT on all 2,000 questions, for those 6 epochs, and regenerate `submission.csv`.

Trade-off to understand: once we train on everything, there is no held-out set left to score *this exact* model. That is fine and standard practice - the Step 8 experiment already validated the approach, so 0.8650 stays our honest estimate. The rule of thumb:

> Use a holdout to **choose** your method, then retrain on **all** the data to **deploy** it.

This cell trains a fresh model from scratch on the full set, then predicts the test set and overwrites `submission.csv`.

**Heads-up:** trains for a few minutes again (now on 2,000 instead of 1,600 questions). Run the cell below.





In [ ]:
# === Step 10: Retrain on ALL 2,000 questions, then rebuild submission.csv ===
# Final model: use every labeled example (no validation holdout).

full_ds = MCQDataset(train, has_labels=True)
full_loader = DataLoader(full_ds, batch_size=TRAIN_BATCH, shuffle=True,
                         num_workers=0, collate_fn=mcq_collate)
# TRAIN_BATCH/ACCUM_STEPS come from Step 8's device-adaptive knobs above,
# so this final retrain gets the same GPU speed-up automatically.
# Train for the epoch count that scored best on validation in Step 8, instead of
# blindly reusing EPOCHS - there is no holdout left here to detect overfitting,
# so the validation run is the only evidence we have about when to stop.
FINAL_EPOCHS = max(BEST_EPOCH, 5)
print(f"Training the final model on the full set: {len(full_ds)} questions "
      f"for {FINAL_EPOCHS} epochs (best epoch from Step 8)\n")

torch.manual_seed(42)
if "model" in globals():        # free the Step-8 model's GPU memory
    del model
torch.cuda.empty_cache()

final_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(device)

if USE_LORA:   # reuses the same lora_config built in Step 8
    final_model = get_peft_model(final_model, lora_config)
    final_model.print_trainable_parameters()

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, final_model.parameters()), lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
final_total_steps = math.ceil(len(full_loader) / ACCUM_STEPS) * FINAL_EPOCHS
final_scheduler = get_linear_schedule_with_warmup(
    optimizer, int(WARMUP_FRAC * final_total_steps), final_total_steps
)

for epoch in range(1, FINAL_EPOCHS + 1):
    final_model.train()
    running = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(full_loader, start=1):
        labels = batch.pop("label").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            out = final_model(**batch, labels=labels)
            loss = out.loss / ACCUM_STEPS
        scaler.scale(loss).backward()
        running += out.loss.item()
        if step % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            final_scheduler.step()
            optimizer.zero_grad()
        if step % 250 == 0:
            print(f"  epoch {epoch}  step {step:4d}/{len(full_loader)}  running loss {running / step:.4f}")
    print(f"epoch {epoch} complete | avg train loss {running / len(full_loader):.4f}")

# ---- Predict the test set with the final model and OVERWRITE submission.csv ----
test_loader = DataLoader(MCQDataset(test, has_labels=False), batch_size=EVAL_BATCH,
                         shuffle=False, num_workers=0, collate_fn=mcq_collate)
final_model.eval()
logits_all = []
with torch.no_grad():
    for batch in test_loader:
        batch.pop("label", None)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = final_model(**batch).logits
        logits_all.append(logits.float().cpu().numpy())

test_logits = np.concatenate(logits_all, axis=0)

# ---- Save the fine-tuned weights BEFORE freeing the model ----
# Without this the trained model is destroyed at the end of every run and there is
# nothing left to deploy. Saving here captures the exact full-data model whose
# predictions produced the submission above.
FINAL_MODEL_DIR = "final_mcq_model"
final_model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Saved the fine-tuned model + tokenizer to {FINAL_MODEL_DIR}/ (for deployment).")

# Optionally push straight to the Hugging Face Hub so a Space can load it.
# Set HF_REPO_ID to "<your-hf-username>/<repo-name>"; leave it empty to skip.
# Needs HF_TOKEN, which Step 0 already reads from Kaggle secrets.
HF_REPO_ID = "ashwinambatwariitm/dlgenai-t22026-mcq"
if HF_REPO_ID:
    _tok = os.environ.get("HF_TOKEN")
    final_model.push_to_hub(HF_REPO_ID, token=_tok)
    tokenizer.push_to_hub(HF_REPO_ID, token=_tok)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")
else:
    print("HF_REPO_ID is empty - skipping the Hub push."
          " Set it to deploy, or download final_mcq_model/ from the Kaggle output.")

# final_model is only needed for the two prediction passes above; everything downstream
# (Step 12) reuses the cached bert_valid_logits_final / test_logits arrays instead, so
# free it now the same way Step 8 freed embed_model and Step 10 freed the Step-8 model.
del final_model
torch.cuda.empty_cache()

ranked = (-test_logits).argsort(axis=1)
top3 = [" ".join(letters[j] for j in row[:3]) for row in ranked]

checkpoint = pd.DataFrame({"ID": test["id"].values, "Prediction": top3})
checkpoint.to_csv("submission_checkpoint_bert.csv", index=False)
print("\nWrote submission_checkpoint_bert.csv (full-data BERT model) | shape:", checkpoint.shape)
print(checkpoint.head(10).to_string(index=False))
print("\nTop-1 predicted letter distribution on test:")
print(checkpoint["Prediction"].str[0].value_counts().reindex(letters).fillna(0).astype(int).to_string())
print("\nThis is a CHECKPOINT file only - Step 12 picks the single best-scoring")
print("model overall (this one, RAG, TF-IDF, scratch, or the ensemble) and writes")
print("THAT to submission.csv. Don't submit this file directly.")

# Log this checkpoint as a W&B artifact (the real submission artifact gets logged in Step 12).
artifact = wandb.Artifact("bert-checkpoint", type="checkpoint")
artifact.add_file("submission_checkpoint_bert.csv")
wandb.log_artifact(artifact)



## Step 11 - RAG: retrieval + causal LM scorer

Design decisions (since there's no external textbook/corpus in this dataset,
only train/test CSVs):
- **Retrieval corpus**: our own labeled questions. For validation scoring we
  retrieve only from `train_split` (never `valid_split`, to avoid leakage);
  for the final submission (Step 11.5) we retrieve from the full `train.csv`.
- **Retriever**: the same MiniLM sentence embeddings from Step 6 (reloaded
  here since Step 8 freed it from GPU memory), used to find the top-k most
  similar *solved* questions to use as few-shot context.
- **Why flat retrieval and not RAPTOR**: RAPTOR builds a recursive tree of
  summaries over a large document collection. That structure pays off when a
  corpus is too big to read directly. Here the corpus is ~2,000 short questions
  with no source documents, so there is no tree worth building and flat top-k
  dense retrieval is the appropriate choice.
- **Causal model**: `Qwen/Qwen2.5-0.5B-Instruct` (small enough to run on the
  competition GPU, instruction-tuned so it handles the few-shot format). For each
  option we compute the causal LM's average per-token log-likelihood of that
  option's text given the retrieved-context + question prompt, and rank
  options by that score - this avoids relying on free-form generation
  producing an invalid answer.




In [ ]:
# === Step 11: RAG - retrieval + causal LM scorer (validation) ===
import numpy as np
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

RAG_TOP_K = 3          # how many similar solved questions to retrieve as few-shot context
CAUSAL_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # swap for a bigger Qwen (1.5B/3B) for more quality, slower
rag_score_key = f"RAG + Causal LM ({CAUSAL_MODEL_NAME})"   # single source of truth for the scoreboard label

retrieval_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
causal_tok = AutoTokenizer.from_pretrained(CAUSAL_MODEL_NAME)
causal_lm = AutoModelForCausalLM.from_pretrained(
    CAUSAL_MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
causal_lm.eval()
if causal_tok.pad_token is None:
    causal_tok.pad_token = causal_tok.eos_token


def build_retrieval_index(corpus_df):
    embeddings = retrieval_model.encode(
        corpus_df["prompt"].astype(str).tolist(), convert_to_tensor=True, show_progress_bar=False
    )
    return corpus_df.reset_index(drop=True), embeddings


def retrieve_examples(query_prompt, corpus_df, corpus_embeddings, k=RAG_TOP_K):
    q_emb = retrieval_model.encode(query_prompt, convert_to_tensor=True, show_progress_bar=False)
    sims = util.cos_sim(q_emb, corpus_embeddings)[0]
    top_idx = torch.topk(sims, k=min(k, len(corpus_df))).indices.tolist()
    return corpus_df.iloc[top_idx]


def build_rag_prompt(row, examples):
    parts = [f"Question: {ex['prompt']}\nAnswer: {ex[ex['answer']]}\n" for _, ex in examples.iterrows()]
    parts.append(f"Question: {row['prompt']}\nAnswer:")
    return "\n".join(parts)


@torch.no_grad()
def option_loglik(prompt_text, option_text):
    """Average per-token log-likelihood of option_text given prompt_text."""
    full_text = prompt_text + " " + option_text
    prompt_ids = causal_tok(prompt_text, return_tensors="pt", truncation=True, max_length=768).input_ids.to(device)
    full_ids = causal_tok(full_text, return_tensors="pt", truncation=True, max_length=800).input_ids.to(device)
    if full_ids.shape[1] <= prompt_ids.shape[1]:
        return -1e9
    out = causal_lm(full_ids)
    log_probs = F.log_softmax(out.logits, dim=-1)
    target_ids = full_ids[0, prompt_ids.shape[1]:]
    token_log_probs = log_probs[0, prompt_ids.shape[1] - 1:-1, :].gather(1, target_ids.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.mean().item()


def rag_score_question(row, corpus_df, corpus_embeddings):
    examples = retrieve_examples(row["prompt"], corpus_df, corpus_embeddings)
    prompt_text = build_rag_prompt(row, examples)
    return [option_loglik(prompt_text, row[c]) for c in option_cols]


def full_rank_from_scores(scores_matrix):
    """(N,5) score array -> list of N full rankings (best-first, letters)."""
    ranked_idx = (-np.asarray(scores_matrix)).argsort(axis=1)
    return [[option_cols[j] for j in row] for row in ranked_idx]


# Retrieval corpus for VALIDATION scoring is train_split only (no leakage from valid_split).
rag_valid_corpus, rag_valid_corpus_emb = build_retrieval_index(train_split)

# --- Diagnostic: near-duplicate questions inflate RAG's score without real reasoning ---
# If a validation question's closest "solved" neighbor is a near-identical question
# (common in template-generated MCQ datasets), the retrieved answer text can leak
# straight into the option list, making RAG look artificially perfect. This just
# measures and reports that risk - it doesn't change any scoring.
NEAR_DUP_THRESHOLD = 0.95
valid_query_embs = retrieval_model.encode(
    valid_split["prompt"].astype(str).tolist(), convert_to_tensor=True, show_progress_bar=False
)
top1_sims = util.cos_sim(valid_query_embs, rag_valid_corpus_emb).max(dim=1).values.cpu().numpy()
near_dup_count = int((top1_sims >= NEAR_DUP_THRESHOLD).sum())
print(
    f"Near-duplicate check: {near_dup_count}/{len(valid_split)} validation questions "
    f"({near_dup_count / len(valid_split):.1%}) have a train_split neighbor with "
    f"cosine similarity >= {NEAR_DUP_THRESHOLD}."
)
if near_dup_count / len(valid_split) > 0.2:
    print(
        "WARNING: a large fraction of validation questions look near-identical to "
        "training questions. If RAG's MAP@3 comes out unusually high (e.g. ~1.0), "
        "treat it with suspicion - it's likely copying retrieved answer text rather "
        "than reasoning, and may not generalize the same way to the real test set."
    )

print(f"Scoring {len(valid_split)} validation questions with retrieval + {CAUSAL_MODEL_NAME} ...")
rag_valid_scores = np.array([
    rag_score_question(row, rag_valid_corpus, rag_valid_corpus_emb)
    for _, row in valid_split.reset_index(drop=True).iterrows()
])

rag_valid_rank = full_rank_from_scores(rag_valid_scores)
rag_valid_preds_top3 = [r[:3] for r in rag_valid_rank]
scoreboard[rag_score_key] = map_at_3(valid_true, rag_valid_preds_top3)
print(f"{rag_score_key} MAP@3: {scoreboard[rag_score_key]:.4f}")
if wandb.run is not None:
    wandb.log({rag_score_key: scoreboard[rag_score_key]})

print("\nScoreboard so far:")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")


## Step 11.5 - (Optional) Submission checkpoint: RAG

Same idea as Steps 5.5/6.6: rebuild the retrieval index over the **full**
`train.csv` (more, and less redundant, few-shot examples to draw from) and
score `test.csv`, writing `submission.csv` with just the RAG model so you can
submit and sanity-check it on its own before the final ensemble.


In [ ]:
# === Step 11.5: Submission checkpoint (RAG) ===
rag_test_corpus, rag_test_corpus_emb = build_retrieval_index(train)  # full labeled data as corpus

print(f"Scoring {len(test)} test questions with retrieval + {CAUSAL_MODEL_NAME} ...")
rag_test_scores = np.array([
    rag_score_question(row, rag_test_corpus, rag_test_corpus_emb)
    for _, row in test.reset_index(drop=True).iterrows()
])
rag_test_rank = full_rank_from_scores(rag_test_scores)
rag_test_top3 = [r[:3] for r in rag_test_rank]

# retrieval_model/causal_lm/causal_tok are only needed for the retrieval + scoring calls
# above; Step 12 reuses the cached rag_valid_rank / rag_test_rank lists instead, so free
# the models now (same pattern as Step 8 freeing embed_model, Step 10 freeing final_model).
del retrieval_model, causal_lm, causal_tok
torch.cuda.empty_cache()

checkpoint = pd.DataFrame({
    "ID": test["id"].values,
    "Prediction": [" ".join(row) for row in rag_test_top3],
})
checkpoint.to_csv("submission_checkpoint_rag.csv", index=False)

print("Wrote submission_checkpoint_rag.csv  | shape:", checkpoint.shape)
print(checkpoint.head(10).to_string(index=False))
print("\nThis is a CHECKPOINT file only, not the real submission - the notebook")
print("picks the single best-scoring model at the very end (Step 12) and writes")
print("THAT to submission.csv. Don'''t submit this file directly.")



## Step 12 - Ensemble all models (weighted Borda count) + final submission

Combines four models' rankings of the 5 options per question - TF-IDF, the
from-scratch BiLSTM, the fine-tuned transformer and RAG - each weighted by its
own validation MAP@3, using a **Borda count**: each model gives
5 points to its top pick, 4 to its second, ... 1 to its last; we sum the
(weighted) points across models and take the top 3 by total. This is a simple
rank-fusion method that doesn't require the different models' raw
scores/logits to be on the same scale (TF-IDF probabilities, BERT logits, and
LM log-likelihoods aren't otherwise comparable).

First we measure the ensemble's own MAP@3 on `valid_split` (using the
train_split-only versions of each model still in memory), then we rebuild the
same ensemble on `test.csv` using each model's full-data-trained version, and
write the final `submission.csv`.



In [ ]:
# === Step 12: Ensemble (weighted Borda count) ===
from collections import defaultdict


def scores_to_full_rank(scores, qids):
    by_q = defaultdict(list)
    for idx, (score, qid) in enumerate(zip(scores, qids)):
        c = option_cols[idx % len(option_cols)]
        by_q[qid].append((c, score))
    return [[c for c, _ in sorted(by_q[qid], key=lambda x: x[1], reverse=True)] for qid in sorted(by_q)]


def borda_points(rankings, weight=1.0):
    n = len(rankings)
    pts = np.zeros((n, len(option_cols)))
    col_index = {c: i for i, c in enumerate(option_cols)}
    for qi, ranking in enumerate(rankings):
        for pos, letter in enumerate(ranking):
            pts[qi, col_index[letter]] += weight * (len(option_cols) - pos)
    return pts


def ensemble_top3(point_matrices):
    total = sum(point_matrices)
    ranked = (-total).argsort(axis=1)
    return [[option_cols[j] for j in row[:3]] for row in ranked]


# ---- 0) Defensive fallback: rebuild anything from optional checkpoint cells
#         (Step 5.5, Step 6.6) if they were skipped in this run, so this cell
#         can'''t crash on a missing variable regardless of what ran before it.
if "best_tfidf_name" not in globals():
    best_tfidf_name = max(
        ["TF-IDF + Logistic Regression", "TF-IDF + SVM"],
        key=lambda name: scoreboard[name],
    )
    print(f"[fallback] Step 5.5 wasn'''t run - best_tfidf_name = {best_tfidf_name}")

if "final_clf" not in globals() or "test_scores" not in globals() or "test_qids" not in globals():
    print("[fallback] Step 5.5 wasn'''t run - refitting TF-IDF on the full train.csv now.")
    full_corpus = train["prompt"].astype(str).tolist()
    for c in option_cols:
        full_corpus += train[c].astype(str).tolist()
    final_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
    final_vectorizer.fit(full_corpus)
    full_texts, full_labels, _ = build_pairwise_rows(train)
    X_full = final_vectorizer.transform(full_texts)
    if best_tfidf_name == "TF-IDF + Logistic Regression":
        final_clf = LogisticRegression(max_iter=1000, class_weight="balanced")
        final_clf.fit(X_full, full_labels)
        score_fn = lambda X: final_clf.predict_proba(X)[:, 1]
    else:
        final_clf = LinearSVC(class_weight="balanced")
        final_clf.fit(X_full, full_labels)
        score_fn = lambda X: final_clf.decision_function(X)
    test_texts, _, test_qids = build_pairwise_rows(test.assign(answer=None))
    X_test = final_vectorizer.transform(test_texts)
    test_scores = score_fn(X_test)

if "final_scratch_model" not in globals() or "final_test_loader" not in globals():
    print("[fallback] Step 6.6 wasn'''t run - retraining the scratch BiLSTM on the full train.csv now.")
    full_counter = Counter()
    for col in ["prompt"] + option_cols:
        for text in train[col].astype(str):
            full_counter.update(simple_tokenize(text))
    final_scratch_vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq in full_counter.items():
        if freq >= MIN_FREQ:
            final_scratch_vocab[word] = len(final_scratch_vocab)

    def final_encode(text, max_len=MAX_LEN_SCRATCH):
        ids = [final_scratch_vocab.get(tok, 1) for tok in simple_tokenize(text)[:max_len]]
        return ids + [0] * (max_len - len(ids))

    class FinalScratchDataset(torch.utils.data.Dataset):
        def __init__(self, df, has_labels=True):
            self.df = df.reset_index(drop=True)
            self.has_labels = has_labels and ("answer" in df.columns)

        def __len__(self):
            return len(self.df)

        def __getitem__(self, i):
            row = self.df.iloc[i]
            prompt_ids = torch.tensor(final_encode(row["prompt"]), dtype=torch.long)
            option_ids = torch.stack(
                [torch.tensor(final_encode(row[c]), dtype=torch.long) for c in option_cols]
            )
            item = {"prompt_ids": prompt_ids, "option_ids": option_ids}
            if self.has_labels:
                item["label"] = torch.tensor(letter_to_idx_scratch[row["answer"]], dtype=torch.long)
            return item

    final_train_ds = FinalScratchDataset(train)
    final_train_loader = torch.utils.data.DataLoader(final_train_ds, batch_size=32, shuffle=True)
    final_scratch_model = ScratchMCQModel(len(final_scratch_vocab)).to(device)
    final_optimizer = torch.optim.Adam(final_scratch_model.parameters(), lr=1e-3)
    for epoch in range(1, SCRATCH_EPOCHS + 1):
        final_scratch_model.train()
        for batch in final_train_loader:
            labels = batch["label"].to(device)
            prompt_ids = batch["prompt_ids"].to(device)
            option_ids = batch["option_ids"].to(device)
            final_optimizer.zero_grad()
            logits = final_scratch_model(prompt_ids, option_ids)
            loss = scratch_criterion(logits, labels)
            loss.backward()
            final_optimizer.step()
    final_test_ds = FinalScratchDataset(test, has_labels=False)
    final_test_loader = torch.utils.data.DataLoader(final_test_ds, batch_size=64, shuffle=False)

if "bert_valid_logits_final" not in globals():
    raise RuntimeError(
        "Step 8 (BERT fine-tuning) hasn'''t run in this session - it must run "
        "before Step 12, since the ensemble needs its cached validation logits."
    )
if "test_logits" not in globals():
    raise RuntimeError("Step 10 (final BERT retrain) hasn'''t run in this session - run it before Step 12.")
if "rag_valid_rank" not in globals() or "rag_test_rank" not in globals():
    raise RuntimeError("Steps 11/11.5 (RAG) haven'''t run in this session - run them before Step 12.")

# ---- 1) Validation ensemble: reuse the train_split-trained models still in memory ----
tfidf_scores_valid = (
    logreg.predict_proba(X_valid)[:, 1]
    if best_tfidf_name == "TF-IDF + Logistic Regression"
    else svm.decision_function(X_valid)
)
tfidf_rank_valid = scores_to_full_rank(tfidf_scores_valid, valid_qids)

scratch_model.eval()
scratch_logits_valid = []
with torch.no_grad():
    for batch in scratch_valid_loader:
        prompt_ids = batch["prompt_ids"].to(device)
        option_ids = batch["option_ids"].to(device)
        scratch_logits_valid.append(scratch_model(prompt_ids, option_ids).cpu().numpy())
scratch_logits_valid = np.concatenate(scratch_logits_valid, axis=0)
scratch_rank_valid = full_rank_from_scores(scratch_logits_valid)

# Step 10 deleted `model` to free GPU memory before training `final_model`, so we reuse
# the validation logits that Step 8'''s evaluate() cached at its BEST epoch (which is
# also the epoch whose score is used as this model's ensemble weight below).
bert_rank_valid = full_rank_from_scores(bert_valid_logits_final)

bert_score_key = max((k for k in scoreboard if k.startswith(model_label)), key=lambda k: scoreboard[k])

valid_components = {
    "TF-IDF": (tfidf_rank_valid, scoreboard[best_tfidf_name]),
    "Scratch BiLSTM": (scratch_rank_valid, scoreboard["From-scratch BiLSTM"]),
    "BERT": (bert_rank_valid, scoreboard[bert_score_key]),
    "RAG": (rag_valid_rank, scoreboard[rag_score_key]),
}
print("Ensemble members and their validation-weighted votes:")
for name, (_, w) in valid_components.items():
    print(f"  {name}: weight (its own MAP@3) = {w:.4f}")

valid_point_matrices = [borda_points(rank, weight=w) for rank, w in valid_components.values()]
ensemble_preds_valid = ensemble_top3(valid_point_matrices)
scoreboard["Ensemble (weighted Borda)"] = map_at_3(valid_true, ensemble_preds_valid)
print(f"\nEnsemble MAP@3 on validation: {scoreboard['Ensemble (weighted Borda)']:.4f}")

print("\nFinal scoreboard:")
for name, score in sorted(scoreboard.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")
if wandb.run is not None:
    wandb.log({"Ensemble (weighted Borda)": scoreboard["Ensemble (weighted Borda)"], "scoreboard_final": scoreboard})

# ---- 2) Test-set ensemble: reuse the full-data-trained models for the real submission ----
tfidf_test_rank = scores_to_full_rank(test_scores, test_qids)  # from Step 5.5 (final_clf on full train)

final_scratch_model.eval()
scratch_logits_test = []
with torch.no_grad():
    for batch in final_test_loader:  # from Step 6.6
        prompt_ids = batch["prompt_ids"].to(device)
        option_ids = batch["option_ids"].to(device)
        scratch_logits_test.append(final_scratch_model(prompt_ids, option_ids).cpu().numpy())
scratch_logits_test = np.concatenate(scratch_logits_test, axis=0)
scratch_test_rank = full_rank_from_scores(scratch_logits_test)

bert_test_rank = full_rank_from_scores(test_logits)  # from Step 10 (final_model on full train)
rag_test_rank_final = rag_test_rank  # from Step 11.5 (retrieval over full train)

test_components = {
    "TF-IDF": (tfidf_test_rank, scoreboard[best_tfidf_name]),
    "Scratch BiLSTM": (scratch_test_rank, scoreboard["From-scratch BiLSTM"]),
    "BERT": (bert_test_rank, scoreboard[bert_score_key]),
    "RAG": (rag_test_rank_final, scoreboard[rag_score_key]),
}
test_point_matrices = [borda_points(rank, weight=w) for rank, w in test_components.values()]
ensemble_test_top3 = ensemble_top3(test_point_matrices)

# ---- 3) Pick ONE overall winner - individual model or the ensemble - and submit ONLY that ----
# This is the only cell in the whole notebook that writes submission.csv; every
# earlier model wrote its own submission_checkpoint_*.csv instead, so nothing
# gets submitted "in between" - only whichever model actually scored best.
final_candidates = {
    best_tfidf_name: [row[:3] for row in tfidf_test_rank],
    "From-scratch BiLSTM": [row[:3] for row in scratch_test_rank],
    bert_score_key: [row[:3] for row in bert_test_rank],
    rag_score_key: [row[:3] for row in rag_test_rank_final],
    "Ensemble (weighted Borda)": ensemble_test_top3,
}
final_candidate_scores = {name: scoreboard[name] for name in final_candidates}

print("Candidates for the final submission (validation MAP@3):")
for name, score in sorted(final_candidate_scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {score:.4f}   {name}")

overall_best_name = max(final_candidate_scores, key=final_candidate_scores.get)
best_test_preds = final_candidates[overall_best_name]

submission = pd.DataFrame({
    "ID": test["id"].values,
    "Prediction": [" ".join(row) for row in best_test_preds],
})
submission.to_csv("submission.csv", index=False)

print(f"\nOVERALL BEST MODEL: '{overall_best_name}' "
      f"(val MAP@3 = {final_candidate_scores[overall_best_name]:.4f})")
print("Wrote FINAL submission.csv using ONLY that model | shape:", submission.shape)
print(submission.head(10).to_string(index=False))
print("\nNext: Save Version -> Submit to Competition in the Kaggle UI,")
print("or:  kaggle competitions submit -c smart-mcq-solver-challenge \\")
print(f"       -f submission.csv -m \"{overall_best_name}\"")

if wandb.run is not None:
    artifact = wandb.Artifact("smart-mcq-submission", type="submission")
    artifact.add_file("submission.csv")
    wandb.log_artifact(artifact)
    wandb.log({"overall_best_model": overall_best_name, "overall_best_score": final_candidate_scores[overall_best_name]})
# NOTE: the run is deliberately NOT finished here. Step 12.5 rewrites submission.csv
# and logs the hybrid retrieval metrics into this same run, so wandb.finish() is
# called at the end of Step 12.5 instead.




## Step 12.5 - Hybrid retrieval layer (memorisation on top of the model)

An EDA finding on the real competition files motivates this step: **the test set
is largely a re-sampling of the same question pool as `train.csv`.**

Measured by exact string comparison rather than embedding similarity, which is
the stricter and more honest test:

| relationship to train | share of test |
|---|---|
| identical prompt **and** identical option set | 51.4% |
| identical prompt only | 2.2% |
| identical option set only | 38.8% |
| no exact match | 7.6% |

An earlier version of this notebook quoted 93.4% based on prompt cosine
similarity >= 0.95. That number was too loose: near-identical wording does not
mean the answer is recoverable, and it led to an over-optimistic score estimate.
The table above supersedes it.

So for a large minority of test questions the answer is directly recoverable:
find the nearest training question, take the **text** of its correct option, and
locate that same text among the current question's options. Using the labelled training set at
inference time is simply a nearest-neighbour model - that is what the labels are
for - but note honestly that it **memorises rather than generalises**. The
fine-tuned model still answers every question retrieval cannot match, and still
supplies ranks 2 and 3 so MAP@3 keeps its partial credit.

One caveat that matters for reading the numbers: Step 4 deliberately removes
near-duplicates from `valid_split`, so this layer shows **no** benefit there *by
construction*. To estimate what it does on the real test set we validate it on a
**random** split instead, because that is how the test set actually relates to
train. Both numbers are reported below and they measure different things:

- **dedup split** -> honest generalisation (what the models really learned)
- **random split** -> expected leaderboard behaviour (given the test set's real composition)



In [ ]:
# === Step 12.5: Hybrid retrieval layer (memorisation on top of the model) ===
# Retrieval answers the questions that literally recur in train.csv; the
# fine-tuned model from Step 12 handles the rest and fills ranks 2-3.
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split as _tts

_rt = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)


def _embed(texts):
    return _rt.encode(list(texts), convert_to_numpy=True, normalize_embeddings=True,
                      show_progress_bar=False, batch_size=64)


def retrieval_answer(target_row, source_row):
    """Take the correct option TEXT from source_row, find that same text among
    target_row's options. Returns the matching letter, or None if the option
    sets differ (in which case we fall back to the model)."""
    answer_text = str(source_row[source_row["answer"]])
    for c in option_cols:
        if str(target_row[c]) == answer_text:
            return c
    return None


# ---- 1) Estimate the benefit on a RANDOM split (this mirrors test -> train) ----
_a, _b = _tts(train, test_size=0.2, random_state=0, stratify=train["answer"])
_a, _b = _a.reset_index(drop=True), _b.reset_index(drop=True)
_nn = (_embed(_b["prompt"].astype(str)) @ _embed(_a["prompt"].astype(str)).T).argmax(axis=1)

_matched, _scores = 0, []
for i in range(len(_b)):
    hit = retrieval_answer(_b.iloc[i], _a.iloc[_nn[i]])
    if hit is None:
        continue
    _matched += 1
    _scores.append(average_precision_at_3(
        _b.iloc[i]["answer"], [hit] + [c for c in option_cols if c != hit]))

print("Random-split check (mirrors how the real test set relates to train):")
print(f"  retrieval matched {_matched}/{len(_b)} questions ({_matched / len(_b):.1%} coverage)")
print(f"  MAP@3 on those matched questions: {np.mean(_scores):.4f}")
print("  (the model keeps its own prediction on everything else)\n")

# ---- 2) Apply to the real test set, retrieving over ALL of train.csv ----
_train_emb = _embed(train["prompt"].astype(str))
_test_emb = _embed(test["prompt"].astype(str))
_test_nn = (_test_emb @ _train_emb.T).argmax(axis=1)

# submission (from Step 12) is built in test order, so row i lines up with test.iloc[i].
model_top3 = [str(p).split() for p in submission["Prediction"].tolist()]

hybrid_preds, n_retrieved, n_changed = [], 0, 0
for i in range(len(test)):
    base = model_top3[i]
    hit = retrieval_answer(test.iloc[i], train.iloc[_test_nn[i]])
    if hit is None:
        ranked = list(base)
    else:
        n_retrieved += 1
        # retrieved answer first, then the model's remaining preferences
        ranked = [hit] + [c for c in base if c != hit]
        ranked += [c for c in option_cols if c not in ranked]
    ranked = ranked[:3]
    if ranked != base:
        n_changed += 1
    hybrid_preds.append(" ".join(ranked))

submission = pd.DataFrame({"ID": test["id"].values, "Prediction": hybrid_preds})
submission.to_csv("submission.csv", index=False)

print(f"Applied to the real test set ({len(test)} questions):")
print(f"  answered by retrieval : {n_retrieved} ({n_retrieved / len(test):.1%})")
print(f"  answered by the model : {len(test) - n_retrieved}")
print(f"  predictions changed vs the model-only submission: {n_changed}")
print("\nWrote the FINAL submission.csv (hybrid: retrieval + "
      f"{overall_best_name}) | shape: {submission.shape}")
print(submission.head(10).to_string(index=False))
print("\nTop-1 predicted letter distribution on test:")
print(submission["Prediction"].str[0].value_counts().reindex(option_cols).fillna(0).astype(int).to_string())

# Free the retriever (same free-when-done pattern used everywhere else in the notebook).
del _rt
if device == "cuda":
    torch.cuda.empty_cache()

if wandb.run is not None:
    wandb.log({
        "hybrid/retrieval_coverage_test": n_retrieved / len(test),
        "hybrid/random_split_coverage": _matched / len(_b),
        "hybrid/random_split_map_at_3_matched": float(np.mean(_scores)),
        "hybrid/predictions_changed": n_changed,
    })
    wandb.summary["hybrid/final_submission"] = f"retrieval + {overall_best_name}"
    artifact = wandb.Artifact("smart-mcq-submission-hybrid", type="submission")
    artifact.add_file("submission.csv")
    wandb.log_artifact(artifact)
    # Close the main pipeline run here (moved from Step 12) so the hybrid metrics
    # above are actually recorded rather than dropped on a finished run.
    wandb.finish()
    print("Logged hybrid metrics + submission artifact, and closed the main W&B run.")



## Step 13 - Log one comparable W&B run per model

Everything above logs into the single `mcq-solver-full-pipeline` run, which is
great for step-by-step training curves (loss, epoch) and artifacts - but a
single run can't be compared against itself in W&B's **Runs** view, since
that view compares *separate* runs against each other.

So here, once every model's final validation MAP@3 is sitting in
`scoreboard`, we open one small run per model (`wandb.init` -> `wandb.log` ->
`wandb.finish`), all logging the *same* metric key `val/map_at_3` and tagged
with `group="model-comparison"`. That gives eleven directly comparable runs
(2 baselines + TF-IDF x2 + MiniLM + three from-scratch encoders + the
fine-tuned transformer + RAG + ensemble), well over the "at least three"
requirement: select them in the Runs sidebar and use the built-in bar chart,
parallel-coordinates or table view to compare every model at a glance.

A final `model-comparison-summary` run then carries the whole comparison as a
sortable `wandb.Table` and a bar chart, plus three panels that need the full
picture: every model's learning curve on shared axes, a confusion matrix over
the option letters, and the leakage measurement from Step 4.



In [ ]:
# === Step 13: Log one comparable W&B run per model ===
# The main run above tracks training curves + artifacts for the whole
# pipeline. This cell instead logs one short run PER MODEL (same metric key
# everywhere: val/map_at_3), so the W&B "Runs" view can compare them
# directly - bar chart, parallel coordinates, or a sortable table. Baselines
# are included too, tagged job_type="baseline" so they read as a floor, not
# as one of the three required models.

comparison_entries = {
    "baseline-constant-abc": ("constant 'A B C'", "baseline"),
    "baseline-most-frequent": (f"constant '{' '.join(freq_order)}' (most frequent)", "baseline"),
    "baseline-answer-text-artifact": (ARTIFACT_BASELINE, "baseline"),
    "tfidf-logreg": ("TF-IDF + Logistic Regression", "classical_ml"),
    "tfidf-svm": ("TF-IDF + SVM", "classical_ml"),
    "minilm-zero-shot": ("Sentence embeddings (MiniLM)", "pretrained_zero_shot"),
    "bilstm-scratch": ("From-scratch BiLSTM", "from_scratch"),
    "cnn-scratch": ("From-scratch text-CNN", "from_scratch"),
    "transformer-scratch": ("From-scratch mini transformer", "from_scratch"),
    f"{BACKBONE}-finetune": (bert_score_key, "pretrained_finetuned"),
    "rag-causal-lm": (rag_score_key, "retrieval_augmented"),
    "ensemble-weighted-borda": ("Ensemble (weighted Borda)", "ensemble"),
}

print("Logging one comparable W&B run per model (group='model-comparison') ...")
for run_name, (scoreboard_key, job_type) in comparison_entries.items():
    if scoreboard_key not in scoreboard:
        print(f"  skipping {run_name}: '{scoreboard_key}' not in scoreboard")
        continue
    wandb.init(
        entity="ambatwar-iitm",
        project="dlgenai-t226",
        name=run_name,
        group="model-comparison",
        job_type=job_type,
        config={"model": scoreboard_key},
    )
    wandb.log({"val/map_at_3": scoreboard[scoreboard_key]})
    wandb.finish()
    print(f"  {run_name:28s} val/map_at_3={scoreboard[scoreboard_key]:.4f}")

# ---- One summary run carrying the comparison table and bar chart ----
# wandb.Table renders as a sortable table panel and wandb.plot.bar as a chart, so
# the complete model comparison is visible in the UI without building panels by hand.
# Only the canonical models belong in the comparison - the same eleven entries the
# per-model runs above use. Iterating `scoreboard` directly would also pull in the
# per-epoch checkpoints ("... after epoch 1", "... after epoch 2", ...), which are
# one model measured ten times, not ten competing models, and they would pad the
# bar chart with duplicate bars.
_comparison_scores = {key: (run_name, job_type)
                      for run_name, (key, job_type) in comparison_entries.items()
                      if key in scoreboard}
summary_rows = [[key, round(float(scoreboard[key]), 5), job_type, rank]
                for rank, (key, (run_name, job_type)) in enumerate(
                    sorted(_comparison_scores.items(),
                           key=lambda kv: scoreboard[kv[0]], reverse=True), start=1)]

wandb.init(
    entity="ambatwar-iitm",
    project="dlgenai-t226",
    name="model-comparison-summary",
    group="model-comparison",
    job_type="summary",
    config={"models_compared": len(summary_rows), "metric": "MAP@3",
            "validation": "leakage-free cluster split, 400 questions"},
)
comparison_table = wandb.Table(data=summary_rows,
                               columns=["model", "val_map_at_3", "category", "rank"])
wandb.log({
    "comparison/table": comparison_table,
    "comparison/bar_chart": wandb.plot.bar(
        comparison_table, "model", "val_map_at_3",
        title="Validation MAP@3 by model (leakage-free split)"),
})
for name, score, _category, _rank in summary_rows:
    wandb.summary[name] = score

# ---- Extra panels: the three things a viva actually asks to see ----
# 1. every trained model's learning curve on ONE axis (not four separate charts)
# 2. which option letter the best model confuses, i.e. is there a positional bias
# 3. hard evidence that the cluster split removed the near-duplicate leakage
# Guarded by existence checks so re-running Step 13 alone never raises, and wrapped
# so a plotting error can never discard the comparison runs logged above.
try:
    _curves = {m: h for m, h in TRAINING_HISTORY.items() if h["epoch"]}
    if _curves:
        _names = list(_curves)
        for _metric, _title in [("val_map_at_3", "Validation MAP@3 per epoch"),
                                ("train_loss", "Training loss per epoch")]:
            wandb.log({f"curves/{_metric}_all_models": wandb.plot.line_series(
                xs=[_curves[m]["epoch"] for m in _names],
                ys=[_curves[m][_metric] for m in _names],
                keys=_names,
                title=f"{_title} - all trained models",
                xname="epoch")})
        print(f"Logged shared learning curves for {len(_names)} trained models: {', '.join(_names)}")

    if globals().get("bert_valid_logits_final") is not None:
        _letters = list(option_cols)
        _true_idx = [_letters.index(a) for a in valid_true]
        _pred_idx = [int(j) for j in bert_valid_logits_final.argmax(axis=1)]
        wandb.log({"analysis/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=_true_idx, preds=_pred_idx, class_names=_letters)})
        _top1 = sum(t == p for t, p in zip(_true_idx, _pred_idx)) / len(_true_idx)
        wandb.summary["analysis/top1_accuracy_best_model"] = _top1
        print(f"Logged a confusion matrix over options {'/'.join(_letters)} "
              f"(top-1 accuracy {_top1:.4f}).")

    if globals().get("LEAKAGE_STATS"):
        _leak_table = wandb.Table(data=[[k, round(v, 4)] for k, v in LEAKAGE_STATS.items()],
                                  columns=["split", "near_duplicate_rate"])
        wandb.log({"leakage/table": _leak_table,
                   "leakage/bar_chart": wandb.plot.bar(
                       _leak_table, "split", "near_duplicate_rate",
                       title="Validation questions with a near-duplicate in train")})
        print("Logged the leakage before/after comparison.")
except Exception as exc:                      # never lose the runs above over a chart
    print(f"WARNING: extra panels skipped ({type(exc).__name__}: {exc})")

wandb.finish()
print(f"\nLogged a summary run with a table + bar chart over all {len(summary_rows)} models.")

print("\nDone. In the W&B UI:")
print("  - Runs table: filter by group == 'model-comparison' and sort by val/map_at_3")
print("  - Charts    : open the 'model-comparison-summary' run for the bar chart,")
print("                the shared learning curves, the confusion matrix and the leakage chart")
print("  - Curves    : open 'mcq-solver-full-pipeline' for per-epoch training curves")

## Step 14 - Combined comparison figures

Steps 4 to 8 already plotted each result where it was produced: the leakage
measurement, one training curve per model, and the confusion matrix. This step
adds the two figures that need every model at once, so they cannot be drawn
earlier:

- all trained models overlaid on shared axes, which is what separates
  architecture from pretraining
- the full ranking of every approach, baselines included

Weights and Biases holds interactive versions of all of these. Everything is
also written to `figures/` as PNGs so the report can embed them directly.


In [ ]:
# === Step 14: Combined comparison figures ===
# Nothing is recomputed here - these read the results the earlier steps produced.
# The per-model curves, confusion matrix and leakage chart were already drawn at
# Steps 4, 6.5, 6.7 and 8, so this step only adds the two all-model views.

# ---- Every trained model overlaid on shared axes ----
# The mini transformer against the fine-tuned backbone is the informative pair:
# same architecture family, so the remaining gap is what pretraining bought.
_curves = {name: hist for name, hist in TRAINING_HISTORY.items() if hist["epoch"]}
if _curves:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for name, hist in _curves.items():
        axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=name)
        axes[1].plot(hist["epoch"], hist["val_map_at_3"], marker="o", label=name)
    axes[0].set_title("Training loss per epoch")
    axes[0].set_ylabel("training loss")
    axes[1].set_title("Validation MAP@3 per epoch")
    axes[1].set_ylabel("MAP@3")
    _epoch_ticks = sorted({e for hist in _curves.values() for e in hist["epoch"]})
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.set_xticks(_epoch_ticks)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle("All trained models on the same axes")
    fig.tight_layout()
    save_and_show(fig, "fig_learning_curves_all_models.png")
else:
    print("no training history in memory - run Steps 6.5, 6.7 and 8 first")

# ---- Every approach ranked, using the same entries as the Step 13 W&B chart ----
_bars = [(run_name, scoreboard[key]) for run_name, (key, _jt) in comparison_entries.items()
         if key in scoreboard]
if _bars:
    _bars.sort(key=lambda kv: kv[1])
    _names = [n for n, _ in _bars]
    _scores = [s for _, s in _bars]
    _colors = ["#b0b0b0" if comparison_entries[n][1] == "baseline" else "#3b7dd8" for n in _names]
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(_bars) + 1.5))
    ax.barh(_names, _scores, color=_colors)
    for y, score in enumerate(_scores):
        ax.text(score + 0.005, y, f"{score:.4f}", va="center", fontsize=8)
    ax.set_xlabel("validation MAP@3 (leakage-free cluster split)")
    ax.set_title("Model comparison - grey bars are baselines, not competing models")
    ax.set_xlim(0, max(_scores) * 1.12)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, "fig_model_comparison.png")

print(f"\n{len(SAVED_FIGURES)} figures available for the report:")
for _path in SAVED_FIGURES:
    print(f"  {_path}")
